In [1]:
# Import Packages
import pandas as pd
import re

# Import from local files
from data_cleaning import DataCleaner
from extraction import extract_data
from entity_resolution import EntityResolver

resolver = EntityResolver()

# Set pandas display options
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)


# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

extracted_df = extract_data(
    "2025 Roll - Parcels Export.xlsx",
    "Only Delinquent Parcels"
)


# ----------------------------------------------------------
# Clean
# ----------------------------------------------------------

cleaner = DataCleaner()

clean_df = cleaner.clean_data(
    extracted_df
)


# ----------------------------------------------------------
# Entity Resolution
# ----------------------------------------------------------

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations
) = EntityResolver().build(
    clean_df
)

# Create audit dataframe.
resolution_audit = (
    entity_occurrences
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address",
                "entity_type",
                "occurrence_count"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one",
        suffixes=("_source", "_final")
    )
)

resolution_audit.head()

,occurrence_id,parcel_number,entity_name,associated_address,role,source_entity_name,source_column,source_occurrence_id,owner_line_continuation_applied,owner_line_continuation_rule,entity_type_source,joint_parse_rule,joint_parse_confidence,joint_expanded,joint_party_index,joint_party_count,organization_type,organization_classification_confidence,address_source,entity_id,canonical_name,canonical_address,entity_type_final,occurrence_count
0,1,01-01-201-013,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Owner,POLOMCAK CLAUDIA,owner_name,1,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,3493,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Person,2
1,2,01-01-251-013,CLAWSON DORRINDA,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA &,owner_name,2,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,953,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,8
2,3,01-01-251-050,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,3,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,953,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,8
3,4,01-01-251-060,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,4,False,,Person,,,False,NaN,NaN,NaN,NaN,owner_address,953,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,8
4,5,01-01-301-050,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Owner,SPIEKERMANN TIM & JUDY,owner_name,5,False,,Person,shared_surname_single_right_name,High,True,1.0,2.0,NaN,NaN,owner_address,4156,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Person,2


In [2]:
entities["occurrence_count"].sum()

np.int64(12067)

In [4]:
kathryn_test_parcels = clean_df[
    clean_df["parcel_number"].isin([
        "02-17-351-053",
        "09-14-410-056"
    ])
]

print(
    kathryn_test_parcels[
        [
            "parcel_number",
            "owner_name",
            "owner_name_2",
            "owner_care_of",
            "taxpayer_name",
            "taxpayer_care_of",
            "tax_send_to_name"
        ]
    ].to_string(index=False)
)

parcel_number            owner_name owner_name_2 owner_care_of         taxpayer_name taxpayer_care_of      tax_send_to_name
02-17-351-053 JONES KATHRYN M TRUST                            JONES KATHRYN M TRUST                  JONES KATHRYN M TRUST


In [5]:
from pathlib import Path
import re
import pandas as pd

from entity_resolution import EntityResolver


def run_stage_1_3_regression_audit(
        clean_df,
        entity_occurrences,
        entities,
        output_dir="audits/stage_1_3_final_regression",
        strict_2025_snapshot=True
):
    """
    Final regression audit for EntityResolver Stages 1-3.

    This function does NOT modify any production DataFrame.

    It validates:
        Stage 1:
            - complete source-cell lineage
            - role/address-source consistency
            - owner-line continuation behavior
            - no dangling ampersands
            - no remaining routing prefixes
            - Joint-expansion integrity

        Stage 2:
            - occurrence/entity provenance remains intact

        Stage 3:
            - every occurrence resolves to exactly one entity
            - occurrence_count reconciliation
            - no duplicate final entity natural keys
            - exact-name Organization shared-parcel duplicates
              have been exhausted
            - safe Organization variant rules have been exhausted
            - intentionally unresolved Organization variants remain
              unresolved
            - Kathryn Jones regression remains separated
            - the known FKA case remains intentionally unresolved

    strict_2025_snapshot=True additionally checks the known-good
    2025 delinquent-tax snapshot totals and manually reviewed cases.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    resolver = EntityResolver()

    checks = []
    metrics = []

    # ==========================================================
    # Helpers
    # ==========================================================

    def add_check(
            name,
            passed,
            details=""
    ):
        checks.append({
            "check": name,
            "status": (
                "PASS"
                if bool(passed)
                else "FAIL"
            ),
            "details": str(details)
        })

    def add_metric(
            name,
            value
    ):
        metrics.append({
            "metric": name,
            "value": value
        })

    def join_unique(values):
        values = [
            str(value)
            for value in values
            if pd.notna(value)
            and str(value).strip() != ""
        ]

        return " | ".join(
            sorted(
                set(values)
            )
        )

    def punctuation_key(name):

        if pd.isna(name):
            return ""

        name = (
            str(name)
            .upper()
            .strip()
        )

        name = re.sub(
            r"[.,]",
            "",
            name
        )

        name = re.sub(
            r"\s+",
            " ",
            name
        ).strip()

        return name

    def compact_key(name):

        if pd.isna(name):
            return ""

        return re.sub(
            r"[^A-Z0-9]",
            "",
            str(name).upper()
        )

    # ==========================================================
    # Basic metrics
    # ==========================================================

    unique_source_occurrences = (
        entity_occurrences[
            "source_occurrence_id"
        ]
        .nunique()
    )

    add_metric(
        "clean_parcels",
        len(clean_df)
    )

    add_metric(
        "unique_source_occurrences",
        unique_source_occurrences
    )

    add_metric(
        "entity_occurrences",
        len(entity_occurrences)
    )

    add_metric(
        "resolved_entities",
        len(entities)
    )

    # ==========================================================
    # STAGE 1
    # Source lineage
    # ==========================================================

    expected_source_parts = []

    for source_column in resolver.role_mapping:

        temp = clean_df[
            [
                "parcel_number",
                source_column
            ]
        ].copy()

        temp[
            source_column
        ] = (
            temp[
                source_column
            ]
            .fillna("")
            .astype(str)
        )

        temp = temp[
            temp[
                source_column
            ]
            .str.strip()
            .ne("")
        ].copy()

        temp[
            "source_column"
        ] = source_column

        temp[
            "source_entity_name"
        ] = temp[
            source_column
        ]

        temp = temp[
            [
                "parcel_number",
                "source_column",
                "source_entity_name"
            ]
        ]

        expected_source_parts.append(
            temp
        )

    expected_source = pd.concat(
        expected_source_parts,
        ignore_index=True
    )

    # Each source_occurrence_id may generate multiple derived rows
    # after Joint decomposition, but all such rows must retain the
    # same source provenance.
    lineage_consistency = (
        entity_occurrences
        .groupby(
            "source_occurrence_id"
        )
        .agg(
            parcel_count=(
                "parcel_number",
                "nunique"
            ),
            source_column_count=(
                "source_column",
                "nunique"
            ),
            source_name_count=(
                "source_entity_name",
                "nunique"
            )
        )
        .reset_index()
    )

    bad_lineage = lineage_consistency[
        (
            lineage_consistency[
                "parcel_count"
            ]
            != 1
        )
        |
        (
            lineage_consistency[
                "source_column_count"
            ]
            != 1
        )
        |
        (
            lineage_consistency[
                "source_name_count"
            ]
            != 1
        )
    ].copy()

    add_check(
        "Every source_occurrence_id has consistent provenance",
        bad_lineage.empty,
        f"bad source occurrence IDs = {len(bad_lineage)}"
    )

    source_level = (
        entity_occurrences
        .sort_values(
            "source_occurrence_id"
        )
        .drop_duplicates(
            "source_occurrence_id"
        )
        [
            [
                "source_occurrence_id",
                "parcel_number",
                "source_column",
                "source_entity_name"
            ]
        ]
        .copy()
    )

    source_comparison = (
        expected_source
        .merge(
            source_level[
                [
                    "parcel_number",
                    "source_column",
                    "source_entity_name"
                ]
            ],
            on=[
                "parcel_number",
                "source_column",
                "source_entity_name"
            ],
            how="outer",
            indicator=True
        )
    )

    source_mismatches = (
        source_comparison[
            source_comparison[
                "_merge"
            ]
            != "both"
        ]
        .copy()
    )

    add_check(
        "Every nonblank source cell is represented exactly once",
        (
            source_mismatches.empty
            and
            len(expected_source)
            ==
            len(source_level)
        ),
        (
            f"expected={len(expected_source)}, "
            f"actual={len(source_level)}, "
            f"mismatches={len(source_mismatches)}"
        )
    )

    # source_occurrence_id should still be a contiguous identity
    # for the original source-cell population.
    expected_source_ids = set(
        range(
            1,
            len(expected_source) + 1
        )
    )

    actual_source_ids = set(
        entity_occurrences[
            "source_occurrence_id"
        ]
        .astype(int)
        .unique()
    )

    add_check(
        "source_occurrence_id values are complete and contiguous",
        actual_source_ids
        ==
        expected_source_ids
    )

    # ==========================================================
    # Stage 1 role lineage
    # ==========================================================

    expected_roles = (
        entity_occurrences[
            "source_column"
        ]
        .map({
            source_column:
                info["role"]
            for source_column, info
            in resolver.role_mapping.items()
        })
    )

    role_mismatches = (
        entity_occurrences[
            expected_roles
            !=
            entity_occurrences[
                "role"
            ]
        ]
        .copy()
    )

    add_check(
        "Occurrence roles agree with source columns",
        role_mismatches.empty,
        f"mismatched rows = {len(role_mismatches)}"
    )

    expected_address_sources = (
        entity_occurrences[
            "role"
        ]
        .map(
            resolver.address_source_mapping
        )
    )

    address_source_mismatches = (
        entity_occurrences[
            expected_address_sources
            !=
            entity_occurrences[
                "address_source"
            ]
        ]
        .copy()
    )

    add_check(
        "address_source agrees with occurrence role",
        address_source_mismatches.empty,
        (
            "mismatched rows = "
            f"{len(address_source_mismatches)}"
        )
    )

    # ==========================================================
    # Stage 1 continuation regression
    # ==========================================================

    continuation_mask = (
        entity_occurrences[
            "owner_line_continuation_applied"
        ]
        .fillna(False)
        .astype(bool)
    )

    continuation_rows = (
        entity_occurrences[
            continuation_mask
        ]
        .copy()
    )

    continuation_parcels = set(
        continuation_rows[
            "parcel_number"
        ]
        .unique()
    )

    add_metric(
        "continuation_rows",
        len(continuation_rows)
    )

    add_metric(
        "continuation_parcels",
        len(continuation_parcels)
    )

    # Every explicitly continued pair should resolve to one
    # working identity representation on that parcel.
    continuation_name_counts = (
        continuation_rows
        .groupby(
            "parcel_number"
        )[
            "entity_name"
        ]
        .nunique()
    )

    add_check(
        "Each continuation parcel has one combined working name",
        (
            continuation_name_counts
            .le(1)
            .all()
        )
    )

    if strict_2025_snapshot:

        expected_continuation_parcels = {
            "06-16-349-005",
            "06-35-147-001",
            "06-35-263-182",
            "06-29-437-025",
            "06-01-205-031",
            "06-35-175-472",
            "06-14-150-003",
            "01-35-170-220",
            "07-05-240-140",
            "07-14-130-012",
            "06-10-342-161",
            "06-10-346-101",
            "06-33-229-001"
        }

        add_check(
            "2025 audited continuation parcel set is unchanged",
            continuation_parcels
            ==
            expected_continuation_parcels,
            (
                "actual="
                f"{sorted(continuation_parcels)}"
            )
        )

        # Previously audited false positives must NEVER be
        # converted into continuation records.
        known_false_positive_parcels = {
            "06-09-477-002",
            "06-23-355-316",
            "06-10-380-022"
        }

        false_positive_continuations = (
            entity_occurrences[
                entity_occurrences[
                    "parcel_number"
                ]
                .isin(
                    known_false_positive_parcels
                )
                &
                continuation_mask
            ]
        )

        add_check(
            "Known continuation false positives remain untouched",
            false_positive_continuations.empty
        )

    # ==========================================================
    # Stage 1 name-cleanup regression
    # ==========================================================

    dangling_ampersands = (
        entity_occurrences[
            entity_occurrences[
                "entity_name"
            ]
            .fillna("")
            .str.contains(
                r"(?:^\s*&)|(?:&\s*$)",
                regex=True
            )
        ]
        .copy()
    )

    add_check(
        "No working entity names have dangling ampersands",
        dangling_ampersands.empty,
        f"rows = {len(dangling_ampersands)}"
    )

    routing_prefix_rows = (
        entity_occurrences[
            entity_occurrences[
                "entity_name"
            ]
            .fillna("")
            .str.contains(
                (
                    r"^\s*(?:"
                    r"C\s*/\s*O\s*:?"
                    r"|ATTN\s*:?"
                    r"|%"
                    r")"
                ),
                regex=True,
                case=False
            )
        ]
        .copy()
    )

    add_check(
        "No working entity names retain routing prefixes",
        routing_prefix_rows.empty,
        f"rows = {len(routing_prefix_rows)}"
    )

    blank_working_names = (
        entity_occurrences[
            entity_occurrences[
                "entity_name"
            ]
            .fillna("")
            .str.strip()
            .eq("")
        ]
    )

    add_check(
        "No Stage 1 working entity names are blank",
        blank_working_names.empty
    )

    # ==========================================================
    # Stage 1 Joint-expansion integrity
    # ==========================================================

    expanded = (
        entity_occurrences[
            entity_occurrences[
                "joint_expanded"
            ]
            .fillna(False)
            .astype(bool)
        ]
        .copy()
    )

    add_metric(
        "joint_expanded_rows",
        len(expanded)
    )

    add_check(
        "Every expanded Joint row resolves to Person",
        (
            expanded.empty
            or
            expanded[
                "entity_type"
            ]
            .eq("Person")
            .all()
        )
    )

    joint_integrity_failures = []

    for source_occurrence_id, group in (
        expanded.groupby(
            "source_occurrence_id"
        )
    ):

        party_counts = (
            group[
                "joint_party_count"
            ]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        if len(party_counts) != 1:

            joint_integrity_failures.append(
                {
                    "source_occurrence_id":
                        source_occurrence_id,
                    "reason":
                        "inconsistent joint_party_count"
                }
            )

            continue

        party_count = party_counts[0]

        indexes = set(
            group[
                "joint_party_index"
            ]
            .dropna()
            .astype(int)
            .tolist()
        )

        expected_indexes = set(
            range(
                1,
                party_count + 1
            )
        )

        if (
            len(group) != party_count
            or indexes != expected_indexes
        ):

            joint_integrity_failures.append(
                {
                    "source_occurrence_id":
                        source_occurrence_id,
                    "reason":
                        (
                            f"rows={len(group)}, "
                            f"party_count={party_count}, "
                            f"indexes={sorted(indexes)}"
                        )
                }
            )

    joint_integrity_failures = pd.DataFrame(
        joint_integrity_failures
    )

    add_check(
        "Joint expansion has complete party indexes/counts",
        joint_integrity_failures.empty,
        (
            "bad source occurrences = "
            f"{len(joint_integrity_failures)}"
        )
    )

    # ==========================================================
    # Stage 3 entity membership
    # ==========================================================

    allowed_entity_types = {
        "Person",
        "Organization",
        "Joint",
        "Unknown"
    }

    unexpected_types = set(
        entities[
            "entity_type"
        ]
        .dropna()
        .unique()
    ) - allowed_entity_types

    add_check(
        "Final entity types are valid",
        not unexpected_types,
        f"unexpected={sorted(unexpected_types)}"
    )

    add_check(
        "entity_id is unique",
        not entities[
            "entity_id"
        ].duplicated().any()
    )

    add_check(
        "Every occurrence has an entity_id",
        not entity_occurrences[
            "entity_id"
        ].isna().any()
    )

    occurrence_entity_ids = set(
        entity_occurrences[
            "entity_id"
        ]
        .dropna()
        .astype(int)
    )

    final_entity_ids = set(
        entities[
            "entity_id"
        ]
        .astype(int)
    )

    add_check(
        "Occurrence/entity ID domains match exactly",
        occurrence_entity_ids
        ==
        final_entity_ids,
        (
            f"occurrence IDs={len(occurrence_entity_ids)}, "
            f"entity IDs={len(final_entity_ids)}"
        )
    )

    # ==========================================================
    # Stage 3 occurrence_count reconciliation
    # ==========================================================

    actual_occurrence_counts = (
        entity_occurrences
        .groupby(
            "entity_id"
        )
        .size()
        .rename(
            "actual_occurrence_count"
        )
        .reset_index()
    )

    occurrence_count_check = (
        entities[
            [
                "entity_id",
                "occurrence_count"
            ]
        ]
        .merge(
            actual_occurrence_counts,
            on="entity_id",
            how="outer"
        )
    )

    occurrence_count_mismatches = (
        occurrence_count_check[
            occurrence_count_check[
                "occurrence_count"
            ]
            !=
            occurrence_count_check[
                "actual_occurrence_count"
            ]
        ]
        .copy()
    )

    add_check(
        "Entity occurrence_count matches Stage 1 occurrences",
        occurrence_count_mismatches.empty,
        (
            "mismatched entities = "
            f"{len(occurrence_count_mismatches)}"
        )
    )

    add_check(
        "Sum of entity occurrence_count equals occurrence rows",
        (
            int(
                entities[
                    "occurrence_count"
                ].sum()
            )
            ==
            len(entity_occurrences)
        )
    )

    # There should never be two final entities with precisely
    # the same natural representation.
    natural_key_frame = (
        entities.copy()
    )

    natural_key_frame[
        "_audit_address"
    ] = (
        natural_key_frame[
            "canonical_address"
        ]
        .fillna("")
    )

    duplicate_natural_keys = (
        natural_key_frame[
            natural_key_frame.duplicated(
                subset=[
                    "canonical_name",
                    "entity_type",
                    "_audit_address"
                ],
                keep=False
            )
        ]
        .copy()
    )

    add_check(
        "No duplicate final entity natural keys remain",
        duplicate_natural_keys.empty,
        f"rows = {len(duplicate_natural_keys)}"
    )

    # ==========================================================
    # Stage 3 Organization audits
    # ==========================================================

    organizations = (
        entities[
            entities[
                "entity_type"
            ]
            ==
            "Organization"
        ]
        .copy()
    )

    organizations[
        "_punctuation_key"
    ] = (
        organizations[
            "canonical_name"
        ]
        .apply(
            punctuation_key
        )
    )

    organizations[
        "_compact_key"
    ] = (
        organizations[
            "canonical_name"
        ]
        .apply(
            compact_key
        )
    )

    # ----------------------------------------------------------
    # Exact Organization names represented by multiple IDs
    # ----------------------------------------------------------

    exact_duplicate_org_summary = (
        organizations
        .groupby(
            "canonical_name",
            sort=True
        )
        .agg(
            entity_count=(
                "entity_id",
                "nunique"
            ),
            entity_ids=(
                "entity_id",
                lambda x:
                    " | ".join(
                        map(
                            str,
                            sorted(
                                set(x)
                            )
                        )
                    )
            ),
            canonical_addresses=(
                "canonical_address",
                join_unique
            ),
            total_occurrences=(
                "occurrence_count",
                "sum"
            )
        )
        .reset_index()
    )

    exact_duplicate_org_summary = (
        exact_duplicate_org_summary[
            exact_duplicate_org_summary[
                "entity_count"
            ]
            > 1
        ]
        .copy()
    )

    add_metric(
        "exact_name_org_duplicate_groups",
        len(
            exact_duplicate_org_summary
        )
    )

    # ----------------------------------------------------------
    # Exact-name Organization duplicates sharing a parcel
    # ----------------------------------------------------------

    org_lookup = (
        organizations[
            [
                "entity_id",
                "canonical_name",
                "_punctuation_key",
                "_compact_key"
            ]
        ]
    )

    org_occurrences = (
        entity_occurrences
        .merge(
            org_lookup,
            on="entity_id",
            how="inner"
        )
    )

    exact_shared_parcel = (
        org_occurrences
        .groupby(
            [
                "parcel_number",
                "canonical_name"
            ]
        )
        .agg(
            entity_count=(
                "entity_id",
                "nunique"
            )
        )
        .reset_index()
    )

    exact_shared_parcel = (
        exact_shared_parcel[
            exact_shared_parcel[
                "entity_count"
            ]
            > 1
        ]
        .copy()
    )

    add_check(
        "No exact-name Organization duplicates share a parcel",
        exact_shared_parcel.empty,
        f"groups = {len(exact_shared_parcel)}"
    )

    # ----------------------------------------------------------
    # Remaining punctuation variants
    # ----------------------------------------------------------

    variant_rows = []

    for key, group in (
        organizations.groupby(
            "_punctuation_key",
            sort=True
        )
    ):

        if (
            group[
                "canonical_name"
            ]
            .nunique()
            <= 1
        ):
            continue

        variant_rows.append({
            "punctuation_key":
                key,
            "entity_count":
                group[
                    "entity_id"
                ].nunique(),
            "canonical_names":
                join_unique(
                    group[
                        "canonical_name"
                    ]
                ),
            "entity_ids":
                " | ".join(
                    map(
                        str,
                        sorted(
                            set(
                                group[
                                    "entity_id"
                                ]
                            )
                        )
                    )
                ),
            "canonical_addresses":
                join_unique(
                    group[
                        "canonical_address"
                    ]
                )
        })

    remaining_org_variants = pd.DataFrame(
        variant_rows
    )

    add_metric(
        "remaining_punctuation_variant_groups",
        len(
            remaining_org_variants
        )
    )

    # Compact-only means names become identical only after
    # removing spacing as well as punctuation.
    compact_only_rows = []

    for key, group in (
        organizations.groupby(
            "_compact_key",
            sort=True
        )
    ):

        if (
            group[
                "canonical_name"
            ]
            .nunique()
            <= 1
        ):
            continue

        if (
            group[
                "_punctuation_key"
            ]
            .nunique()
            <= 1
        ):
            # Already a punctuation variant, not compact-only.
            continue

        compact_only_rows.append({
            "compact_key":
                key,
            "canonical_names":
                join_unique(
                    group[
                        "canonical_name"
                    ]
                ),
            "canonical_addresses":
                join_unique(
                    group[
                        "canonical_address"
                    ]
                )
        })

    compact_only_variants = pd.DataFrame(
        compact_only_rows
    )

    add_check(
        "No compact/spacing-only Organization variants remain",
        compact_only_variants.empty,
        (
            "groups = "
            f"{len(compact_only_variants)}"
        )
    )

    # ----------------------------------------------------------
    # Variant Organizations sharing a parcel
    # ----------------------------------------------------------

    punctuation_shared_parcel = (
        org_occurrences
        .groupby(
            [
                "parcel_number",
                "_punctuation_key"
            ]
        )
        .agg(
            entity_count=(
                "entity_id",
                "nunique"
            ),
            distinct_name_count=(
                "canonical_name",
                "nunique"
            )
        )
        .reset_index()
    )

    punctuation_shared_parcel = (
        punctuation_shared_parcel[
            (
                punctuation_shared_parcel[
                    "entity_count"
                ]
                > 1
            )
            &
            (
                punctuation_shared_parcel[
                    "distinct_name_count"
                ]
                > 1
            )
        ]
        .copy()
    )

    add_check(
        "No punctuation-variant Organizations share a parcel",
        punctuation_shared_parcel.empty,
        (
            "groups = "
            f"{len(punctuation_shared_parcel)}"
        )
    )

    compact_shared_parcel = (
        org_occurrences
        .groupby(
            [
                "parcel_number",
                "_compact_key"
            ]
        )
        .agg(
            entity_count=(
                "entity_id",
                "nunique"
            ),
            distinct_name_count=(
                "canonical_name",
                "nunique"
            )
        )
        .reset_index()
    )

    compact_shared_parcel = (
        compact_shared_parcel[
            (
                compact_shared_parcel[
                    "entity_count"
                ]
                > 1
            )
            &
            (
                compact_shared_parcel[
                    "distinct_name_count"
                ]
                > 1
            )
        ]
        .copy()
    )

    add_check(
        "No compact-name Organization variants share a parcel",
        compact_shared_parcel.empty,
        (
            "groups = "
            f"{len(compact_shared_parcel)}"
        )
    )

    # ----------------------------------------------------------
    # No safe exact-address variant should remain unresolved
    # ----------------------------------------------------------

    exact_address_variant_groups = []

    for _, group in (
        organizations.groupby(
            [
                "organization_type",
                "_punctuation_key",
                "canonical_address"
            ],
            dropna=False,
            sort=False
        )
    ):

        if (
            len(group) > 1
            and
            group[
                "canonical_name"
            ]
            .nunique()
            > 1
        ):
            exact_address_variant_groups.append(
                group
            )

    add_check(
        "No punctuation-name variants remain at exact same address",
        len(
            exact_address_variant_groups
        )
        == 0,
        (
            "groups = "
            f"{len(exact_address_variant_groups)}"
        )
    )

    # ==========================================================
    # Known intentionally unresolved FKA case
    # ==========================================================

    fka_entities = (
        entities[
            entities[
                "canonical_name"
            ]
            .fillna("")
            .str.contains(
                r"^\s*\(\s*FKA\b",
                regex=True,
                case=False
            )
        ]
        .copy()
    )

    add_metric(
        "remaining_fka_entities",
        len(fka_entities)
    )

    if strict_2025_snapshot:

        add_check(
            "Exactly one known FKA entity remains intentionally unresolved",
            (
                len(fka_entities)
                == 1
            )
        )

        if len(fka_entities) == 1:

            fka_id = (
                fka_entities.iloc[0][
                    "entity_id"
                ]
            )

            fka_occurrences = (
                entity_occurrences[
                    entity_occurrences[
                        "entity_id"
                    ]
                    ==
                    fka_id
                ]
                .copy()
            )

            add_check(
                "Known FKA case is still the reviewed Foley/Nizzardini record",
                (
                    set(
                        fka_occurrences[
                            "parcel_number"
                        ]
                    )
                    ==
                    {
                        "06-33-372-016"
                    }
                    and
                    set(
                        fka_occurrences[
                            "source_column"
                        ]
                    )
                    ==
                    {
                        "owner_name_2"
                    }
                    and
                    set(
                        fka_occurrences[
                            "source_entity_name"
                        ]
                    )
                    ==
                    {
                        "(FKA NIZZARDINI, TERI)"
                    }
                )
            )

        else:
            fka_occurrences = pd.DataFrame()

    else:
        fka_occurrences = pd.DataFrame()

    # ==========================================================
    # Kathryn Jones false-identity regression
    # ==========================================================

    # ==========================================================
    # Kathryn Jones false-identity regression
    # ==========================================================

    kathryn_trust_parcel = "02-17-351-053"
    kathryn_person_reference_parcel = "09-14-410-056"

    # ----------------------------------------------------------
    # Identify entities attached to the known trust parcel.
    # ----------------------------------------------------------

    trust_parcel_occurrences = (
        entity_occurrences[
            entity_occurrences[
                "parcel_number"
            ]
            ==
            kathryn_trust_parcel
        ]
        .copy()
    )

    trust_parcel_entity_ids = set(
        trust_parcel_occurrences[
            "entity_id"
        ]
        .dropna()
        .astype(int)
    )

    trust_parcel_entities = (
        entities[
            entities[
                "entity_id"
            ]
            .isin(
                trust_parcel_entity_ids
            )
        ]
        .copy()
    )

    # Specifically identify Kathryn/Jones entities on that parcel.
    kathryn_trust_entities = (
        trust_parcel_entities[
            trust_parcel_entities[
                "canonical_name"
            ]
            .fillna("")
            .str.upper()
            .str.contains(
                "KATHRYN",
                regex=False
            )
            &
            trust_parcel_entities[
                "canonical_name"
            ]
            .fillna("")
            .str.upper()
            .str.contains(
                "JONES",
                regex=False
            )
        ]
        .copy()
    )

    if strict_2025_snapshot:

        add_check(
            "Kathryn Jones trust parcel is represented",
            not kathryn_trust_entities.empty,
            (
                "matching entities="
                f"{kathryn_trust_entities['entity_id'].tolist()}"
            )
        )

        add_check(
            "Kathryn Jones trust identity is not classified as Person",
            (
                not kathryn_trust_entities.empty
                and
                not kathryn_trust_entities[
                    "entity_type"
                ]
                .eq("Person")
                .any()
            ),
            (
                "types="
                f"{kathryn_trust_entities['entity_type'].tolist()}"
            )
        )

    # ----------------------------------------------------------
    # Make sure the trust's entity ID has not absorbed a Person
    # Kathryn Jones elsewhere in the CURRENT delinquent dataset.
    # ----------------------------------------------------------

    all_kathryn_jones_people = (
        entities[
            (
                entities[
                    "entity_type"
                ]
                ==
                "Person"
            )
            &
            entities[
                "canonical_name"
            ]
            .fillna("")
            .str.upper()
            .str.contains(
                "KATHRYN",
                regex=False
            )
            &
            entities[
                "canonical_name"
            ]
            .fillna("")
            .str.upper()
            .str.contains(
                "JONES",
                regex=False
            )
        ]
        .copy()
    )

    trust_entity_ids = set(
        kathryn_trust_entities[
            "entity_id"
        ]
        .astype(int)
    )

    person_entity_ids = set(
        all_kathryn_jones_people[
            "entity_id"
        ]
        .astype(int)
    )

    add_check(
        "Kathryn Jones trust identity is separate from Person Kathryn Jones entities",
        trust_entity_ids.isdisjoint(
            person_entity_ids
        ),
        (
            "shared entity IDs="
            f"{sorted(trust_entity_ids & person_entity_ids)}"
        )
    )

    # ----------------------------------------------------------
    # Optional comparison-parcel check.
    #
    # The known Person parcel came from the broader source context
    # and may not be present in Only Delinquent Parcels.
    # If it IS present, enforce separation. If it is absent, that
    # is not a regression failure.
    # ----------------------------------------------------------

    reference_parcel_present = (
        clean_df[
            "parcel_number"
        ]
        .eq(
            kathryn_person_reference_parcel
        )
        .any()
    )

    add_metric(
        "kathryn_person_reference_parcel_present",
        bool(
            reference_parcel_present
        )
    )

    if reference_parcel_present:

        reference_ids = set(
            entity_occurrences.loc[
                entity_occurrences[
                    "parcel_number"
                ]
                ==
                kathryn_person_reference_parcel,
                "entity_id"
            ]
            .dropna()
            .astype(int)
        )

        add_check(
            "Kathryn Jones reference parcel remains separate from trust parcel",
            trust_entity_ids.isdisjoint(
                reference_ids
            ),
            (
                "shared entity IDs="
                f"{sorted(trust_entity_ids & reference_ids)}"
            )
        )

    # ==========================================================
    # 2025 snapshot regression fixtures
    # ==========================================================

    if strict_2025_snapshot:

        add_check(
            "2025 parcel count",
            len(clean_df)
            == 3874,
            f"actual={len(clean_df)}"
        )

        add_check(
            "2025 source occurrence count",
            unique_source_occurrences
            == 10270,
            f"actual={unique_source_occurrences}"
        )

        add_check(
            "2025 entity occurrence count",
            len(entity_occurrences)
            == 12067,
            f"actual={len(entity_occurrences)}"
        )

        add_check(
            "2025 resolved entity count",
            len(entities)
            == 4954,
            f"actual={len(entities)}"
        )

        expected_remaining_variant_keys = {
            "907 HOMECREST LLC",
            "ABSOLUTE HEAVEN LLC",
            "BANGKA PROPERTIES LLC",
            "BLACK SWAN BUILDERS LLC",
            "CHOOSE OUR HOME LLC",
            "DREAM MANAGEMENT LLC",
            "MIKA FIRST LLC",
            "STATE WIDE RENTALS LLC",
            "YOUNG & YOUNG LLC"
        }

        actual_remaining_variant_keys = set(
            remaining_org_variants[
                "punctuation_key"
            ]
            if not remaining_org_variants.empty
            else []
        )

        add_check(
            "2025 intentionally unresolved Organization variant set is unchanged",
            actual_remaining_variant_keys
            ==
            expected_remaining_variant_keys,
            (
                "actual="
                f"{sorted(actual_remaining_variant_keys)}"
            )
        )

    # ==========================================================
    # Export audit artifacts
    # ==========================================================

    checks_df = pd.DataFrame(
        checks
    )

    metrics_df = pd.DataFrame(
        metrics
    )

    checks_df.to_csv(
        output_dir
        /
        "stage_1_3_regression_checks.csv",
        index=False
    )

    metrics_df.to_csv(
        output_dir
        /
        "stage_1_3_regression_metrics.csv",
        index=False
    )

    continuation_rows.to_csv(
        output_dir
        /
        "continuation_rows.csv",
        index=False
    )

    exact_duplicate_org_summary.to_csv(
        output_dir
        /
        "remaining_exact_org_duplicates.csv",
        index=False
    )

    remaining_org_variants.to_csv(
        output_dir
        /
        "remaining_org_variants.csv",
        index=False
    )

    source_mismatches.to_csv(
        output_dir
        /
        "source_lineage_mismatches.csv",
        index=False
    )

    duplicate_natural_keys.to_csv(
        output_dir
        /
        "duplicate_natural_keys.csv",
        index=False
    )

    exact_shared_parcel.to_csv(
        output_dir
        /
        "exact_org_shared_parcel_failures.csv",
        index=False
    )

    punctuation_shared_parcel.to_csv(
        output_dir
        /
        "punctuation_variant_shared_parcel_failures.csv",
        index=False
    )


    if not fka_occurrences.empty:

        fka_occurrences.to_csv(
            output_dir
            /
            "known_fka_exception.csv",
            index=False
        )

    # ==========================================================
    # Console summary
    # ==========================================================

    print(
        "\n"
        "=========================================="
    )

    print(
        "FINAL STAGE 1-3 REGRESSION AUDIT"
    )

    print(
        "=========================================="
    )

    print(
        f"\nSource occurrences: "
        f"{unique_source_occurrences:,}"
    )

    print(
        f"Entity occurrences: "
        f"{len(entity_occurrences):,}"
    )

    print(
        f"Resolved entities: "
        f"{len(entities):,}"
    )

    print(
        f"Continuation parcels: "
        f"{len(continuation_parcels):,}"
    )

    print(
        f"Remaining exact-name Organization "
        f"duplicate groups: "
        f"{len(exact_duplicate_org_summary):,}"
    )

    print(
        f"Remaining Organization variant groups: "
        f"{len(remaining_org_variants):,}"
    )

    print(
        f"Remaining FKA exceptions: "
        f"{len(fka_entities):,}"
    )

    print(
        "\nChecks:"
    )

    print(
        checks_df[
            [
                "status",
                "check"
            ]
        ]
        .to_string(
            index=False
        )
    )

    failed_checks = (
        checks_df[
            checks_df[
                "status"
            ]
            ==
            "FAIL"
        ]
    )

    print()

    if failed_checks.empty:

        print(
            "RESULT: PASS"
        )

        print(
            "Stages 1-3 passed the final regression audit."
        )

    else:

        print(
            f"RESULT: FAIL "
            f"({len(failed_checks)} checks failed)"
        )

        print()

        print(
            failed_checks[
                [
                    "check",
                    "details"
                ]
            ]
            .to_string(
                index=False
            )
        )

        raise AssertionError(
            "Stage 1-3 regression audit failed. "
            f"See {output_dir} for audit files."
        )

    print(
        "\nAudit files written to:"
    )

    print(
        output_dir.resolve()
    )

    return (
        checks_df,
        metrics_df
    )


# ==============================================================
# RUN AUDIT
# ==============================================================

stage_1_3_checks, stage_1_3_metrics = (
    run_stage_1_3_regression_audit(
        clean_df=clean_df,
        entity_occurrences=entity_occurrences,
        entities=entities,
        strict_2025_snapshot=True
    )
)


FINAL STAGE 1-3 REGRESSION AUDIT

Source occurrences: 10,270
Entity occurrences: 12,067
Resolved entities: 4,954
Continuation parcels: 13
Remaining exact-name Organization duplicate groups: 17
Remaining Organization variant groups: 9
Remaining FKA exceptions: 1

Checks:
status                                                                       check
  PASS                        Every source_occurrence_id has consistent provenance
  PASS                      Every nonblank source cell is represented exactly once
  PASS                     source_occurrence_id values are complete and contiguous
  PASS                                  Occurrence roles agree with source columns
  PASS                                  address_source agrees with occurrence role
  PASS                      Each continuation parcel has one combined working name
  PASS                           2025 audited continuation parcel set is unchanged
  PASS                         Known continuation false positive

In [6]:
from pathlib import Path

import pandas as pd


def run_stage_4_6_regression_audit(
        clean_df,
        entity_occurrences,
        entities,
        parcel_entity_roles,
        association_evidence,
        entity_associations,
        output_dir="audits/stage_4_6_final_regression",
        strict_2025_snapshot=True
):
    """
    Final regression audit for EntityResolver Stages 4-6.

    This audit does NOT modify any production DataFrame.

    Stage 4:
        - validates parcel/entity/role grain
        - verifies all references are valid
        - reconstructs Stage 4 directly from entity_occurrences

    Stage 5:
        - validates evidence IDs and normalized entity pairs
        - validates approved evidence rules and strengths
        - validates same-parcel evidence against Stage 4
        - validates rule-specific Organization/Person roles
        - validates shared-address evidence metadata
        - checks for duplicate evidence rows
        - checks that continuation cleanup did not recreate
          false co-owner evidence
        - checks the Kathryn Jones false-identity regression

    Stage 6:
        - validates one row per entity pair
        - requires exact pair correspondence with Stage 5
        - reconciles evidence_count
        - reconciles evidence_rule_count
        - verifies association_confidence equals the strongest
          individual evidence strength
        - confirms evidence quantity never boosts confidence

    strict_2025_snapshot=True additionally checks the known-good
    2025 row counts and approved evidence-rule set.
    """

    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    checks = []
    metrics = []

    # ==========================================================
    # Helpers
    # ==========================================================

    def add_check(
            name,
            passed,
            details=""
    ):
        checks.append({
            "check": name,
            "status": (
                "PASS"
                if bool(passed)
                else "FAIL"
            ),
            "details": str(details)
        })

    def add_metric(
            name,
            value
    ):
        metrics.append({
            "metric": name,
            "value": value
        })

    def is_blank(value):
        return (
            pd.isna(value)
            or str(value).strip() == ""
        )

    # ==========================================================
    # Approved values
    # ==========================================================

    valid_roles = {
        "Owner",
        "Owner Care Of",
        "Taxpayer",
        "Taxpayer Care Of",
        "Tax Send To"
    }

    valid_strengths = {
        "High",
        "Medium",
        "Low"
    }

    strength_rank = {
        "Low": 1,
        "Medium": 2,
        "High": 3
    }

    # ----------------------------------------------------------
    # Approved Stage 5 evidence rules.
    #
    # These encode the current v1 semantics and therefore form
    # part of the regression contract.
    # ----------------------------------------------------------

    rule_strengths = {
        "SAME_PARCEL_CO_OWNER":
            "High",

        "SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER":
            "High",

        "SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER":
            "High",

        "SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO":
            "Medium",

        "SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER":
            "Medium",

        "SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF":
            "Low",

        "SHARED_OWNER_ADDRESS":
            "Medium",

        "SHARED_TAXPAYER_ADDRESS":
            "Medium"
    }

    same_parcel_rules = {
        "SAME_PARCEL_CO_OWNER",
        "SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER",
        "SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER",
        "SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO",
        "SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER",
        "SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF"
    }

    shared_address_rules = {
        "SHARED_OWNER_ADDRESS",
        "SHARED_TAXPAYER_ADDRESS"
    }

    # ==========================================================
    # Basic metrics
    # ==========================================================

    add_metric(
        "parcel_entity_roles",
        len(parcel_entity_roles)
    )

    add_metric(
        "association_evidence",
        len(association_evidence)
    )

    add_metric(
        "entity_associations",
        len(entity_associations)
    )

    add_metric(
        "distinct_evidence_pairs",
        association_evidence[
            [
                "entity_id_a",
                "entity_id_b"
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )

    # ==========================================================
    # Shared reference domains
    # ==========================================================

    valid_entity_ids = set(
        entities[
            "entity_id"
        ].astype(int)
    )

    valid_parcels = set(
        clean_df[
            "parcel_number"
        ]
    )

    entity_type_lookup = dict(
        zip(
            entities[
                "entity_id"
            ],
            entities[
                "entity_type"
            ]
        )
    )

    entity_name_lookup = dict(
        zip(
            entities[
                "entity_id"
            ],
            entities[
                "canonical_name"
            ]
        )
    )

    # ==========================================================
    # STAGE 4
    # Parcel Entity Roles
    # ==========================================================

    duplicate_role_rows = (
        parcel_entity_roles[
            parcel_entity_roles.duplicated(
                subset=[
                    "parcel_number",
                    "entity_id",
                    "role"
                ],
                keep=False
            )
        ]
        .copy()
    )

    add_check(
        "Stage 4 has one row per parcel/entity/role",
        duplicate_role_rows.empty,
        (
            "duplicate rows = "
            f"{len(duplicate_role_rows)}"
        )
    )

    invalid_stage4_roles = (
        parcel_entity_roles[
            ~parcel_entity_roles[
                "role"
            ]
            .isin(
                valid_roles
            )
        ]
        .copy()
    )

    add_check(
        "Stage 4 contains only approved roles",
        invalid_stage4_roles.empty,
        (
            "invalid rows = "
            f"{len(invalid_stage4_roles)}"
        )
    )

    invalid_stage4_entity_ids = (
        set(
            parcel_entity_roles[
                "entity_id"
            ].astype(int)
        )
        -
        valid_entity_ids
    )

    add_check(
        "Stage 4 references only finalized entities",
        not invalid_stage4_entity_ids,
        (
            "invalid entity IDs = "
            f"{sorted(invalid_stage4_entity_ids)}"
        )
    )

    invalid_stage4_parcels = (
        set(
            parcel_entity_roles[
                "parcel_number"
            ]
        )
        -
        valid_parcels
    )

    add_check(
        "Stage 4 references only source parcels",
        not invalid_stage4_parcels,
        (
            "invalid parcels = "
            f"{sorted(invalid_stage4_parcels)}"
        )
    )

    # ----------------------------------------------------------
    # Reconstruct Stage 4 directly from Stage 1-3 output.
    #
    # Stage 4 should be exactly:
    #
    #   distinct(parcel_number, entity_id, role)
    # ----------------------------------------------------------

    expected_stage4 = (
        entity_occurrences[
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    actual_stage4 = (
        parcel_entity_roles[
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    stage4_comparison = (
        expected_stage4
        .merge(
            actual_stage4,
            on=[
                "parcel_number",
                "entity_id",
                "role"
            ],
            how="outer",
            indicator=True
        )
    )

    stage4_mismatches = (
        stage4_comparison[
            stage4_comparison[
                "_merge"
            ]
            != "both"
        ]
        .copy()
    )

    add_check(
        "Stage 4 exactly matches distinct Stage 1-3 parcel/entity/role rows",
        (
            stage4_mismatches.empty
            and
            len(expected_stage4)
            ==
            len(parcel_entity_roles)
        ),
        (
            f"expected={len(expected_stage4)}, "
            f"actual={len(parcel_entity_roles)}, "
            f"mismatches={len(stage4_mismatches)}"
        )
    )

    # ==========================================================
    # STAGE 5
    # Association Evidence
    # ==========================================================

    # ----------------------------------------------------------
    # Evidence IDs
    # ----------------------------------------------------------

    add_check(
        "Stage 5 evidence_id is unique",
        association_evidence[
            "evidence_id"
        ].is_unique
    )

    expected_evidence_ids = set(
        range(
            1,
            len(association_evidence) + 1
        )
    )

    actual_evidence_ids = set(
        association_evidence[
            "evidence_id"
        ]
        .astype(int)
    )

    add_check(
        "Stage 5 evidence_id values are complete and contiguous",
        actual_evidence_ids
        ==
        expected_evidence_ids
    )

    # ----------------------------------------------------------
    # Normalized entity pairs
    # ----------------------------------------------------------

    normalized_evidence_pairs = (
        association_evidence[
            "entity_id_a"
        ]
        <
        association_evidence[
            "entity_id_b"
        ]
    )

    add_check(
        "Stage 5 evidence pairs are normalized and non-self-referencing",
        normalized_evidence_pairs.all(),
        (
            "bad rows = "
            f"{(~normalized_evidence_pairs).sum()}"
        )
    )

    # ----------------------------------------------------------
    # Entity-domain validation
    # ----------------------------------------------------------

    evidence_entity_ids = (
        set(
            association_evidence[
                "entity_id_a"
            ].astype(int)
        )
        |
        set(
            association_evidence[
                "entity_id_b"
            ].astype(int)
        )
    )

    invalid_evidence_entity_ids = (
        evidence_entity_ids
        -
        valid_entity_ids
    )

    add_check(
        "Stage 5 references only finalized entities",
        not invalid_evidence_entity_ids,
        (
            "invalid entity IDs = "
            f"{sorted(invalid_evidence_entity_ids)}"
        )
    )

    # ----------------------------------------------------------
    # Parcel-domain validation
    # ----------------------------------------------------------

    evidence_parcels = set()

    for column in [
        "parcel_number_a",
        "parcel_number_b"
    ]:

        evidence_parcels.update(
            value
            for value
            in association_evidence[
                column
            ]
            if not is_blank(value)
        )

    invalid_evidence_parcels = (
        evidence_parcels
        -
        valid_parcels
    )

    add_check(
        "Stage 5 references only source parcels",
        not invalid_evidence_parcels,
        (
            "invalid parcels = "
            f"{sorted(invalid_evidence_parcels)}"
        )
    )

    # ----------------------------------------------------------
    # Role-domain validation
    # ----------------------------------------------------------

    invalid_evidence_roles = (
        association_evidence[
            ~association_evidence[
                "role_a"
            ].isin(
                valid_roles
            )
            |
            ~association_evidence[
                "role_b"
            ].isin(
                valid_roles
            )
        ]
        .copy()
    )

    add_check(
        "Stage 5 contains only approved endpoint roles",
        invalid_evidence_roles.empty,
        (
            "invalid rows = "
            f"{len(invalid_evidence_roles)}"
        )
    )

    # ----------------------------------------------------------
    # Evidence-rule validation
    # ----------------------------------------------------------

    actual_rules = set(
        association_evidence[
            "evidence_rule"
        ]
        .dropna()
        .unique()
    )

    unexpected_rules = (
        actual_rules
        -
        set(
            rule_strengths
        )
    )

    add_check(
        "Stage 5 contains only approved evidence rules",
        not unexpected_rules,
        (
            "unexpected rules = "
            f"{sorted(unexpected_rules)}"
        )
    )

    # ----------------------------------------------------------
    # Strength-domain validation
    # ----------------------------------------------------------

    invalid_strength_rows = (
        association_evidence[
            ~association_evidence[
                "evidence_strength"
            ]
            .isin(
                valid_strengths
            )
        ]
        .copy()
    )

    add_check(
        "Stage 5 contains only approved evidence strengths",
        invalid_strength_rows.empty,
        (
            "invalid rows = "
            f"{len(invalid_strength_rows)}"
        )
    )

    # ----------------------------------------------------------
    # Rule -> strength regression contract
    # ----------------------------------------------------------

    expected_strength = (
        association_evidence[
            "evidence_rule"
        ]
        .map(
            rule_strengths
        )
    )

    rule_strength_mismatches = (
        association_evidence[
            expected_strength.notna()
            &
            (
                association_evidence[
                    "evidence_strength"
                ]
                !=
                expected_strength
            )
        ]
        .copy()
    )

    add_check(
        "Each evidence rule retains its approved strength",
        rule_strength_mismatches.empty,
        (
            "mismatched rows = "
            f"{len(rule_strength_mismatches)}"
        )
    )

    # ----------------------------------------------------------
    # Exact duplicate evidence rows
    #
    # Repeated evidence for one pair/rule can be legitimate when
    # it comes from distinct parcel/support contexts.
    #
    # Only completely duplicate evidence rows are prohibited.
    # ----------------------------------------------------------

    evidence_payload_columns = [
        column
        for column
        in association_evidence.columns
        if column != "evidence_id"
    ]

    duplicate_evidence_rows = (
        association_evidence[
            association_evidence.duplicated(
                subset=evidence_payload_columns,
                keep=False
            )
        ]
        .copy()
    )

    add_check(
        "Stage 5 contains no exact duplicate evidence rows",
        duplicate_evidence_rows.empty,
        (
            "duplicate rows = "
            f"{len(duplicate_evidence_rows)}"
        )
    )

    # ==========================================================
    # Stage 5 Organization/Person pair validation
    # ==========================================================

    bad_type_pairs = []

    for row in association_evidence.itertuples(
            index=False
    ):

        type_a = entity_type_lookup.get(
            row.entity_id_a
        )

        type_b = entity_type_lookup.get(
            row.entity_id_b
        )

        if {
            type_a,
            type_b
        } != {
            "Organization",
            "Person"
        }:

            bad_type_pairs.append({
                "evidence_id":
                    row.evidence_id,

                "entity_id_a":
                    row.entity_id_a,

                "entity_type_a":
                    type_a,

                "entity_id_b":
                    row.entity_id_b,

                "entity_type_b":
                    type_b,

                "evidence_rule":
                    row.evidence_rule
            })

    bad_type_pairs = pd.DataFrame(
        bad_type_pairs
    )

    add_check(
        "Every Stage 5 evidence pair links one Organization and one Person",
        bad_type_pairs.empty,
        (
            "bad rows = "
            f"{len(bad_type_pairs)}"
        )
    )

    # ==========================================================
    # Same-parcel evidence metadata
    # ==========================================================

    same_parcel_evidence = (
        association_evidence[
            association_evidence[
                "evidence_rule"
            ]
            .isin(
                same_parcel_rules
            )
        ]
        .copy()
    )

    same_parcel_metadata_failures = []

    for row in same_parcel_evidence.itertuples(
            index=False
    ):

        valid = (
            not is_blank(
                row.parcel_number_a
            )
            and
            not is_blank(
                row.parcel_number_b
            )
            and
            row.parcel_number_a
            ==
            row.parcel_number_b
            and
            is_blank(
                row.shared_address
            )
            and
            is_blank(
                row.address_type
            )
        )

        if not valid:

            same_parcel_metadata_failures.append(
                {
                    "evidence_id":
                        row.evidence_id,

                    "evidence_rule":
                        row.evidence_rule,

                    "parcel_number_a":
                        row.parcel_number_a,

                    "parcel_number_b":
                        row.parcel_number_b,

                    "shared_address":
                        row.shared_address,

                    "address_type":
                        row.address_type
                }
            )

    same_parcel_metadata_failures = pd.DataFrame(
        same_parcel_metadata_failures
    )

    add_check(
        "Same-parcel evidence has consistent parcel metadata",
        same_parcel_metadata_failures.empty,
        (
            "bad rows = "
            f"{len(same_parcel_metadata_failures)}"
        )
    )

    # ==========================================================
    # Same-parcel evidence must be supported by Stage 4 roles
    # ==========================================================

    stage4_role_set = set(
        parcel_entity_roles[
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        ]
        .itertuples(
            index=False,
            name=None
        )
    )

    unsupported_same_parcel_evidence = []

    for row in same_parcel_evidence.itertuples(
            index=False
    ):

        if (
            is_blank(
                row.parcel_number_a
            )
            or
            row.parcel_number_a
            !=
            row.parcel_number_b
        ):
            continue

        parcel_number = (
            row.parcel_number_a
        )

        endpoint_a = (
            parcel_number,
            row.entity_id_a,
            row.role_a
        )

        endpoint_b = (
            parcel_number,
            row.entity_id_b,
            row.role_b
        )

        missing = []

        if endpoint_a not in stage4_role_set:
            missing.append(
                "A"
            )

        if endpoint_b not in stage4_role_set:
            missing.append(
                "B"
            )

        if missing:

            unsupported_same_parcel_evidence.append({
                "evidence_id":
                    row.evidence_id,

                "evidence_rule":
                    row.evidence_rule,

                "parcel_number":
                    parcel_number,

                "entity_id_a":
                    row.entity_id_a,

                "role_a":
                    row.role_a,

                "entity_id_b":
                    row.entity_id_b,

                "role_b":
                    row.role_b,

                "missing_endpoint":
                    " + ".join(
                        missing
                    )
            })

    unsupported_same_parcel_evidence = pd.DataFrame(
        unsupported_same_parcel_evidence
    )

    add_check(
        "Every same-parcel evidence endpoint is supported by Stage 4",
        unsupported_same_parcel_evidence.empty,
        (
            "unsupported rows = "
            f"{len(unsupported_same_parcel_evidence)}"
        )
    )

    # ==========================================================
    # Rule-specific Organization/Person role semantics
    # ==========================================================

    expected_rule_roles = {
        "SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER":
            (
                "Owner",
                "Taxpayer"
            ),

        "SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER":
            (
                "Taxpayer",
                "Owner"
            ),

        "SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO":
            (
                "Owner",
                "Tax Send To"
            ),

        "SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER":
            (
                "Tax Send To",
                "Owner"
            ),

        "SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF":
            (
                "Owner",
                "Owner Care Of"
            )
    }

    rule_semantic_failures = []

    for row in same_parcel_evidence.itertuples(
            index=False
    ):

        type_a = entity_type_lookup.get(
            row.entity_id_a
        )

        type_b = entity_type_lookup.get(
            row.entity_id_b
        )

        # ------------------------------------------------------
        # Co-owner rule:
        # both endpoints must be Owners.
        # ------------------------------------------------------

        if (
            row.evidence_rule
            ==
            "SAME_PARCEL_CO_OWNER"
        ):

            if not (
                row.role_a == "Owner"
                and
                row.role_b == "Owner"
            ):

                rule_semantic_failures.append({
                    "evidence_id":
                        row.evidence_id,

                    "evidence_rule":
                        row.evidence_rule,

                    "type_a":
                        type_a,

                    "role_a":
                        row.role_a,

                    "type_b":
                        type_b,

                    "role_b":
                        row.role_b
                })

            continue

        # ------------------------------------------------------
        # All remaining same-parcel rules have explicitly
        # oriented Organization and Person roles.
        # ------------------------------------------------------

        if (
            row.evidence_rule
            not in
            expected_rule_roles
        ):
            continue

        (
            expected_org_role,
            expected_person_role
        ) = expected_rule_roles[
            row.evidence_rule
        ]

        if type_a == "Organization":

            org_role = row.role_a
            person_role = row.role_b

        elif type_b == "Organization":

            org_role = row.role_b
            person_role = row.role_a

        else:

            rule_semantic_failures.append({
                "evidence_id":
                    row.evidence_id,

                "evidence_rule":
                    row.evidence_rule,

                "type_a":
                    type_a,

                "role_a":
                    row.role_a,

                "type_b":
                    type_b,

                "role_b":
                    row.role_b
            })

            continue

        if not (
            org_role
            ==
            expected_org_role
            and
            person_role
            ==
            expected_person_role
        ):

            rule_semantic_failures.append({
                "evidence_id":
                    row.evidence_id,

                "evidence_rule":
                    row.evidence_rule,

                "type_a":
                    type_a,

                "role_a":
                    row.role_a,

                "type_b":
                    type_b,

                "role_b":
                    row.role_b,

                "expected_org_role":
                    expected_org_role,

                "expected_person_role":
                    expected_person_role
            })

    rule_semantic_failures = pd.DataFrame(
        rule_semantic_failures
    )

    add_check(
        "Same-parcel evidence obeys rule-specific role semantics",
        rule_semantic_failures.empty,
        (
            "bad rows = "
            f"{len(rule_semantic_failures)}"
        )
    )

    # ==========================================================
    # Shared-address evidence metadata
    # ==========================================================

    shared_address_evidence = (
        association_evidence[
            association_evidence[
                "evidence_rule"
            ]
            .isin(
                shared_address_rules
            )
        ]
        .copy()
    )

    shared_address_metadata_failures = []

    expected_address_type = {
        "SHARED_OWNER_ADDRESS":
            "Owner",

        "SHARED_TAXPAYER_ADDRESS":
            "Taxpayer"
    }

    for row in shared_address_evidence.itertuples(
            index=False
    ):

        valid = (
            is_blank(
                row.parcel_number_a
            )
            and
            is_blank(
                row.parcel_number_b
            )
            and
            not is_blank(
                row.shared_address
            )
            and
            row.address_type
            ==
            expected_address_type[
                row.evidence_rule
            ]
        )

        if not valid:

            shared_address_metadata_failures.append({
                "evidence_id":
                    row.evidence_id,

                "evidence_rule":
                    row.evidence_rule,

                "parcel_number_a":
                    row.parcel_number_a,

                "parcel_number_b":
                    row.parcel_number_b,

                "shared_address":
                    row.shared_address,

                "address_type":
                    row.address_type
            })

    shared_address_metadata_failures = pd.DataFrame(
        shared_address_metadata_failures
    )

    add_check(
        "Shared-address evidence has consistent address metadata",
        shared_address_metadata_failures.empty,
        (
            "bad rows = "
            f"{len(shared_address_metadata_failures)}"
        )
    )

    # ==========================================================
    # Continuation regression
    #
    # Explicit continuation cases must not reappear downstream
    # as SAME_PARCEL_CO_OWNER evidence.
    # ==========================================================

    continuation_parcels = set(
        entity_occurrences.loc[
            entity_occurrences[
                "owner_line_continuation_applied"
            ]
            .fillna(False)
            .astype(bool),
            "parcel_number"
        ]
    )

    continuation_co_owner_failures = (
        association_evidence[
            (
                association_evidence[
                    "evidence_rule"
                ]
                ==
                "SAME_PARCEL_CO_OWNER"
            )
            &
            (
                association_evidence[
                    "parcel_number_a"
                ]
                .isin(
                    continuation_parcels
                )
                |
                association_evidence[
                    "parcel_number_b"
                ]
                .isin(
                    continuation_parcels
                )
            )
        ]
        .copy()
    )

    add_check(
        "Continuation parcels do not recreate false co-owner evidence",
        continuation_co_owner_failures.empty,
        (
            "bad rows = "
            f"{len(continuation_co_owner_failures)}"
        )
    )

    # ==========================================================
    # Kathryn Jones downstream regression
    # ==========================================================

    kathryn_trust_parcel = (
        "02-17-351-053"
    )

    kathryn_trust_occurrences = (
        entity_occurrences[
            entity_occurrences[
                "parcel_number"
            ]
            ==
            kathryn_trust_parcel
        ]
        .copy()
    )

    kathryn_trust_entity_ids = set()

    for entity_id in (
        kathryn_trust_occurrences[
            "entity_id"
        ]
        .dropna()
        .astype(int)
        .unique()
    ):

        name = str(
            entity_name_lookup.get(
                entity_id,
                ""
            )
        ).upper()

        if (
            "KATHRYN" in name
            and
            "JONES" in name
        ):
            kathryn_trust_entity_ids.add(
                entity_id
            )

    kathryn_person_entity_ids = set(
        entities.loc[
            (
                entities[
                    "entity_type"
                ]
                ==
                "Person"
            )
            &
            entities[
                "canonical_name"
            ]
            .fillna("")
            .str.upper()
            .str.contains(
                "KATHRYN",
                regex=False
            )
            &
            entities[
                "canonical_name"
            ]
            .fillna("")
            .str.upper()
            .str.contains(
                "JONES",
                regex=False
            ),
            "entity_id"
        ]
        .astype(int)
    )

    prohibited_kathryn_pairs = {
        tuple(
            sorted(
                (
                    trust_id,
                    person_id
                )
            )
        )
        for trust_id
        in kathryn_trust_entity_ids
        for person_id
        in kathryn_person_entity_ids
        if trust_id
        !=
        person_id
    }

    evidence_pair_set = set(
        association_evidence[
            [
                "entity_id_a",
                "entity_id_b"
            ]
        ]
        .itertuples(
            index=False,
            name=None
        )
    )

    kathryn_evidence_failures = (
        prohibited_kathryn_pairs
        &
        evidence_pair_set
    )

    add_check(
        "Kathryn Jones false identity is not recreated in Stage 5 evidence",
        not kathryn_evidence_failures,
        (
            "prohibited evidence pairs = "
            f"{sorted(kathryn_evidence_failures)}"
        )
    )

    # ==========================================================
    # STAGE 6
    # Entity Associations
    # ==========================================================

    add_check(
        "Stage 6 association_id is unique",
        entity_associations[
            "association_id"
        ].is_unique
    )

    expected_association_ids = set(
        range(
            1,
            len(entity_associations) + 1
        )
    )

    actual_association_ids = set(
        entity_associations[
            "association_id"
        ]
        .astype(int)
    )

    add_check(
        "Stage 6 association_id values are complete and contiguous",
        actual_association_ids
        ==
        expected_association_ids
    )

    normalized_association_pairs = (
        entity_associations[
            "entity_id_a"
        ]
        <
        entity_associations[
            "entity_id_b"
        ]
    )

    add_check(
        "Stage 6 association pairs are normalized and non-self-referencing",
        normalized_association_pairs.all(),
        (
            "bad rows = "
            f"{(~normalized_association_pairs).sum()}"
        )
    )

    duplicate_association_pairs = (
        entity_associations[
            entity_associations.duplicated(
                subset=[
                    "entity_id_a",
                    "entity_id_b"
                ],
                keep=False
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 contains exactly one row per entity pair",
        duplicate_association_pairs.empty,
        (
            "duplicate rows = "
            f"{len(duplicate_association_pairs)}"
        )
    )

    association_entity_ids = (
        set(
            entity_associations[
                "entity_id_a"
            ]
            .astype(int)
        )
        |
        set(
            entity_associations[
                "entity_id_b"
            ]
            .astype(int)
        )
    )

    invalid_association_entity_ids = (
        association_entity_ids
        -
        valid_entity_ids
    )

    add_check(
        "Stage 6 references only finalized entities",
        not invalid_association_entity_ids,
        (
            "invalid entity IDs = "
            f"{sorted(invalid_association_entity_ids)}"
        )
    )

    invalid_confidences = (
        entity_associations[
            ~entity_associations[
                "association_confidence"
            ]
            .isin(
                valid_strengths
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 contains only approved confidence values",
        invalid_confidences.empty,
        (
            "invalid rows = "
            f"{len(invalid_confidences)}"
        )
    )

    # ==========================================================
    # Stage 5 <-> Stage 6 pair-set equality
    # ==========================================================

    association_pair_set = set(
        entity_associations[
            [
                "entity_id_a",
                "entity_id_b"
            ]
        ]
        .itertuples(
            index=False,
            name=None
        )
    )

    missing_stage6_pairs = (
        evidence_pair_set
        -
        association_pair_set
    )

    unsupported_stage6_pairs = (
        association_pair_set
        -
        evidence_pair_set
    )

    add_check(
        "Every Stage 5 evidence pair exists in Stage 6",
        not missing_stage6_pairs,
        (
            "missing pairs = "
            f"{sorted(missing_stage6_pairs)}"
        )
    )

    add_check(
        "Every Stage 6 association has Stage 5 evidence",
        not unsupported_stage6_pairs,
        (
            "unsupported pairs = "
            f"{sorted(unsupported_stage6_pairs)}"
        )
    )

    # ==========================================================
    # Reconstruct Stage 6 entirely from Stage 5
    # ==========================================================

    expected_association_rows = []

    for (
        entity_id_a,
        entity_id_b
    ), group in association_evidence.groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        sort=False
    ):

        evidence_count = len(
            group
        )

        evidence_rule_count = (
            group[
                "evidence_rule"
            ]
            .nunique()
        )

        strongest_strength = max(
            group[
                "evidence_strength"
            ],
            key=lambda value:
                strength_rank[
                    value
                ]
        )

        expected_association_rows.append({
            "entity_id_a":
                entity_id_a,

            "entity_id_b":
                entity_id_b,

            "expected_evidence_count":
                evidence_count,

            "expected_evidence_rule_count":
                evidence_rule_count,

            "expected_association_confidence":
                strongest_strength
        })

    expected_associations = pd.DataFrame(
        expected_association_rows
    )

    association_reconciliation = (
        entity_associations
        .merge(
            expected_associations,
            on=[
                "entity_id_a",
                "entity_id_b"
            ],
            how="outer",
            indicator=True
        )
    )

    association_reconciliation_failures = (
        association_reconciliation[
            (
                association_reconciliation[
                    "_merge"
                ]
                != "both"
            )
            |
            (
                association_reconciliation[
                    "evidence_count"
                ]
                !=
                association_reconciliation[
                    "expected_evidence_count"
                ]
            )
            |
            (
                association_reconciliation[
                    "evidence_rule_count"
                ]
                !=
                association_reconciliation[
                    "expected_evidence_rule_count"
                ]
            )
            |
            (
                association_reconciliation[
                    "association_confidence"
                ]
                !=
                association_reconciliation[
                    "expected_association_confidence"
                ]
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 exactly reconciles to Stage 5 evidence",
        association_reconciliation_failures.empty,
        (
            "bad associations = "
            f"{len(association_reconciliation_failures)}"
        )
    )

    # These individual checks make the console output more
    # diagnostic if the combined reconciliation ever fails.

    evidence_count_mismatches = (
        association_reconciliation[
            (
                association_reconciliation[
                    "_merge"
                ]
                ==
                "both"
            )
            &
            (
                association_reconciliation[
                    "evidence_count"
                ]
                !=
                association_reconciliation[
                    "expected_evidence_count"
                ]
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 evidence_count equals actual evidence-row count",
        evidence_count_mismatches.empty,
        (
            "mismatches = "
            f"{len(evidence_count_mismatches)}"
        )
    )

    evidence_rule_count_mismatches = (
        association_reconciliation[
            (
                association_reconciliation[
                    "_merge"
                ]
                ==
                "both"
            )
            &
            (
                association_reconciliation[
                    "evidence_rule_count"
                ]
                !=
                association_reconciliation[
                    "expected_evidence_rule_count"
                ]
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 evidence_rule_count equals distinct Stage 5 rules",
        evidence_rule_count_mismatches.empty,
        (
            "mismatches = "
            f"{len(evidence_rule_count_mismatches)}"
        )
    )

    confidence_mismatches = (
        association_reconciliation[
            (
                association_reconciliation[
                    "_merge"
                ]
                ==
                "both"
            )
            &
            (
                association_reconciliation[
                    "association_confidence"
                ]
                !=
                association_reconciliation[
                    "expected_association_confidence"
                ]
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 confidence equals strongest individual evidence strength",
        confidence_mismatches.empty,
        (
            "mismatches = "
            f"{len(confidence_mismatches)}"
        )
    )

    # ----------------------------------------------------------
    # Count-domain rules
    # ----------------------------------------------------------

    invalid_association_counts = (
        entity_associations[
            (
                entity_associations[
                    "evidence_count"
                ]
                < 1
            )
            |
            (
                entity_associations[
                    "evidence_rule_count"
                ]
                < 1
            )
            |
            (
                entity_associations[
                    "evidence_rule_count"
                ]
                >
                entity_associations[
                    "evidence_count"
                ]
            )
        ]
        .copy()
    )

    add_check(
        "Stage 6 evidence counts satisfy count constraints",
        invalid_association_counts.empty,
        (
            "bad rows = "
            f"{len(invalid_association_counts)}"
        )
    )

    add_check(
        "Stage 6 total evidence_count reconciles to Stage 5",
        (
            int(
                entity_associations[
                    "evidence_count"
                ].sum()
            )
            ==
            len(
                association_evidence
            )
        ),
        (
            "Stage 6 sum="
            f"{entity_associations['evidence_count'].sum()}, "
            "Stage 5 rows="
            f"{len(association_evidence)}"
        )
    )

    # ==========================================================
    # Kathryn Jones Stage 6 check
    # ==========================================================

    kathryn_association_failures = (
        prohibited_kathryn_pairs
        &
        association_pair_set
    )

    add_check(
        "Kathryn Jones false identity is not recreated in Stage 6 associations",
        not kathryn_association_failures,
        (
            "prohibited association pairs = "
            f"{sorted(kathryn_association_failures)}"
        )
    )

    # ==========================================================
    # Audit summaries
    # ==========================================================

    evidence_rule_counts = (
        association_evidence[
            "evidence_rule"
        ]
        .value_counts()
        .rename_axis(
            "evidence_rule"
        )
        .reset_index(
            name="evidence_count"
        )
    )

    evidence_strength_counts = (
        association_evidence[
            "evidence_strength"
        ]
        .value_counts()
        .rename_axis(
            "evidence_strength"
        )
        .reset_index(
            name="evidence_count"
        )
    )

    association_confidence_counts = (
        entity_associations[
            "association_confidence"
        ]
        .value_counts()
        .rename_axis(
            "association_confidence"
        )
        .reset_index(
            name="association_count"
        )
    )

    evidence_count_distribution = (
        entity_associations[
            "evidence_count"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "evidence_count"
        )
        .reset_index(
            name="association_count"
        )
    )

    rule_count_distribution = (
        entity_associations[
            "evidence_rule_count"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "evidence_rule_count"
        )
        .reset_index(
            name="association_count"
        )
    )

    # ----------------------------------------------------------
    # Pair-level explanatory summary
    # ----------------------------------------------------------

    pair_support_summary = (
        association_evidence
        .groupby(
            [
                "entity_id_a",
                "entity_id_b"
            ]
        )
        .agg(
            evidence_count=(
                "evidence_id",
                "size"
            ),
            evidence_rule_count=(
                "evidence_rule",
                "nunique"
            ),
            evidence_rules=(
                "evidence_rule",
                lambda values:
                    " + ".join(
                        sorted(
                            set(
                                values
                            )
                        )
                    )
            ),
            evidence_strengths=(
                "evidence_strength",
                lambda values:
                    " + ".join(
                        sorted(
                            set(
                                values
                            ),
                            key=lambda value:
                                -strength_rank[
                                    value
                                ]
                        )
                    )
            )
        )
        .reset_index()
    )

    pair_support_summary = (
        pair_support_summary
        .merge(
            entity_associations[
                [
                    "association_id",
                    "entity_id_a",
                    "entity_id_b",
                    "association_confidence"
                ]
            ],
            on=[
                "entity_id_a",
                "entity_id_b"
            ],
            how="left"
        )
    )

    pair_support_summary[
        "entity_name_a"
    ] = (
        pair_support_summary[
            "entity_id_a"
        ]
        .map(
            entity_name_lookup
        )
    )

    pair_support_summary[
        "entity_name_b"
    ] = (
        pair_support_summary[
            "entity_id_b"
        ]
        .map(
            entity_name_lookup
        )
    )

    # ==========================================================
    # Snapshot regression checks
    # ==========================================================

    if strict_2025_snapshot:

        add_check(
            "2025 Stage 4 row count",
            len(
                parcel_entity_roles
            )
            ==
            12050,
            (
                "actual="
                f"{len(parcel_entity_roles)}"
            )
        )

        add_check(
            "2025 Stage 5 evidence row count",
            len(
                association_evidence
            )
            ==
            477,
            (
                "actual="
                f"{len(association_evidence)}"
            )
        )

        add_check(
            "2025 Stage 6 association row count",
            len(
                entity_associations
            )
            ==
            299,
            (
                "actual="
                f"{len(entity_associations)}"
            )
        )

        add_check(
            "2025 approved evidence-rule set is unchanged",
            actual_rules
            ==
            set(
                rule_strengths
            ),
            (
                "actual="
                f"{sorted(actual_rules)}"
            )
        )

        add_check(
            "2025 continuation parcel count remains 13",
            len(
                continuation_parcels
            )
            ==
            13,
            (
                "actual="
                f"{len(continuation_parcels)}"
            )
        )

        add_check(
            "2025 resolved-entity baseline remains 4,954",
            len(
                entities
            )
            ==
            4954,
            (
                "actual="
                f"{len(entities)}"
            )
        )

    # ==========================================================
    # Metrics from final distributions
    # ==========================================================

    add_metric(
        "high_associations",
        int(
            (
                entity_associations[
                    "association_confidence"
                ]
                ==
                "High"
            ).sum()
        )
    )

    add_metric(
        "medium_associations",
        int(
            (
                entity_associations[
                    "association_confidence"
                ]
                ==
                "Medium"
            ).sum()
        )
    )

    add_metric(
        "low_associations",
        int(
            (
                entity_associations[
                    "association_confidence"
                ]
                ==
                "Low"
            ).sum()
        )
    )

    add_metric(
        "multi_evidence_associations",
        int(
            (
                entity_associations[
                    "evidence_count"
                ]
                >
                1
            ).sum()
        )
    )

    add_metric(
        "multi_rule_associations",
        int(
            (
                entity_associations[
                    "evidence_rule_count"
                ]
                >
                1
            ).sum()
        )
    )

    add_metric(
        "max_evidence_count",
        int(
            entity_associations[
                "evidence_count"
            ].max()
        )
        if not entity_associations.empty
        else 0
    )

    add_metric(
        "max_evidence_rule_count",
        int(
            entity_associations[
                "evidence_rule_count"
            ].max()
        )
        if not entity_associations.empty
        else 0
    )

    # ==========================================================
    # Export audit artifacts
    # ==========================================================

    checks_df = pd.DataFrame(
        checks
    )

    metrics_df = pd.DataFrame(
        metrics
    )

    checks_df.to_csv(
        output_dir
        /
        "stage_4_6_regression_checks.csv",
        index=False
    )

    metrics_df.to_csv(
        output_dir
        /
        "stage_4_6_regression_metrics.csv",
        index=False
    )

    evidence_rule_counts.to_csv(
        output_dir
        /
        "evidence_rule_counts.csv",
        index=False
    )

    evidence_strength_counts.to_csv(
        output_dir
        /
        "evidence_strength_counts.csv",
        index=False
    )

    association_confidence_counts.to_csv(
        output_dir
        /
        "association_confidence_counts.csv",
        index=False
    )

    evidence_count_distribution.to_csv(
        output_dir
        /
        "evidence_count_distribution.csv",
        index=False
    )

    rule_count_distribution.to_csv(
        output_dir
        /
        "evidence_rule_count_distribution.csv",
        index=False
    )

    pair_support_summary.to_csv(
        output_dir
        /
        "association_pair_support_summary.csv",
        index=False
    )

    stage4_mismatches.to_csv(
        output_dir
        /
        "stage4_reconstruction_mismatches.csv",
        index=False
    )

    duplicate_role_rows.to_csv(
        output_dir
        /
        "duplicate_stage4_roles.csv",
        index=False
    )

    duplicate_evidence_rows.to_csv(
        output_dir
        /
        "duplicate_evidence_rows.csv",
        index=False
    )

    rule_strength_mismatches.to_csv(
        output_dir
        /
        "rule_strength_mismatches.csv",
        index=False
    )

    bad_type_pairs.to_csv(
        output_dir
        /
        "bad_entity_type_pairs.csv",
        index=False
    )

    same_parcel_metadata_failures.to_csv(
        output_dir
        /
        "same_parcel_metadata_failures.csv",
        index=False
    )

    unsupported_same_parcel_evidence.to_csv(
        output_dir
        /
        "unsupported_same_parcel_evidence.csv",
        index=False
    )

    rule_semantic_failures.to_csv(
        output_dir
        /
        "rule_semantic_failures.csv",
        index=False
    )

    shared_address_metadata_failures.to_csv(
        output_dir
        /
        "shared_address_metadata_failures.csv",
        index=False
    )

    continuation_co_owner_failures.to_csv(
        output_dir
        /
        "continuation_co_owner_failures.csv",
        index=False
    )

    association_reconciliation_failures.to_csv(
        output_dir
        /
        "stage6_reconciliation_failures.csv",
        index=False
    )

    evidence_count_mismatches.to_csv(
        output_dir
        /
        "evidence_count_mismatches.csv",
        index=False
    )

    evidence_rule_count_mismatches.to_csv(
        output_dir
        /
        "evidence_rule_count_mismatches.csv",
        index=False
    )

    confidence_mismatches.to_csv(
        output_dir
        /
        "association_confidence_mismatches.csv",
        index=False
    )

    # ==========================================================
    # Console summary
    # ==========================================================

    print(
        "\n"
        "=========================================="
    )

    print(
        "FINAL STAGE 4-6 REGRESSION AUDIT"
    )

    print(
        "=========================================="
    )

    print(
        f"\nParcel-entity-role rows: "
        f"{len(parcel_entity_roles):,}"
    )

    print(
        f"Association evidence rows: "
        f"{len(association_evidence):,}"
    )

    print(
        f"Entity associations: "
        f"{len(entity_associations):,}"
    )

    print(
        f"Distinct evidence pairs: "
        f"{len(evidence_pair_set):,}"
    )

    print(
        "\nEvidence rules:"
    )

    print(
        evidence_rule_counts
        .to_string(
            index=False
        )
    )

    print(
        "\nEvidence strengths:"
    )

    print(
        evidence_strength_counts
        .to_string(
            index=False
        )
    )

    print(
        "\nAssociation confidence:"
    )

    print(
        association_confidence_counts
        .to_string(
            index=False
        )
    )

    print(
        "\nEvidence-count distribution:"
    )

    print(
        evidence_count_distribution
        .to_string(
            index=False
        )
    )

    print(
        "\nChecks:"
    )

    print(
        checks_df[
            [
                "status",
                "check"
            ]
        ]
        .to_string(
            index=False
        )
    )

    failed_checks = (
        checks_df[
            checks_df[
                "status"
            ]
            ==
            "FAIL"
        ]
    )

    print()

    if failed_checks.empty:

        print(
            "RESULT: PASS"
        )

        print(
            "Stages 4-6 passed the final regression audit."
        )

    else:

        print(
            f"RESULT: FAIL "
            f"({len(failed_checks)} checks failed)"
        )

        print()

        print(
            failed_checks[
                [
                    "check",
                    "details"
                ]
            ]
            .to_string(
                index=False
            )
        )

        raise AssertionError(
            "Stage 4-6 regression audit failed. "
            f"See {output_dir} for audit files."
        )

    print(
        "\nAudit files written to:"
    )

    print(
        output_dir.resolve()
    )

    return (
        checks_df,
        metrics_df
    )


# ==============================================================
# RUN AUDIT
# ==============================================================

stage_4_6_checks, stage_4_6_metrics = (
    run_stage_4_6_regression_audit(
        clean_df=clean_df,
        entity_occurrences=entity_occurrences,
        entities=entities,
        parcel_entity_roles=parcel_entity_roles,
        association_evidence=association_evidence,
        entity_associations=entity_associations,
        strict_2025_snapshot=True
    )
)


FINAL STAGE 4-6 REGRESSION AUDIT

Parcel-entity-role rows: 12,050
Association evidence rows: 477
Entity associations: 299
Distinct evidence pairs: 299

Evidence rules:
                             evidence_rule  evidence_count
                      SHARED_OWNER_ADDRESS              91
     SAME_PARCEL_ORG_OWNER_PERSON_TAXPAYER              79
                   SHARED_TAXPAYER_ADDRESS              63
     SAME_PARCEL_ORG_TAXPAYER_PERSON_OWNER              55
                      SAME_PARCEL_CO_OWNER              49
  SAME_PARCEL_ORG_OWNER_PERSON_TAX_SEND_TO              49
  SAME_PARCEL_ORG_TAX_SEND_TO_PERSON_OWNER              49
SAME_PARCEL_ORG_OWNER_PERSON_OWNER_CARE_OF              42

Evidence strengths:
evidence_strength  evidence_count
           Medium             252
             High             183
              Low              42

Association confidence:
association_confidence  association_count
                  High                147
                Medium            

In [7]:
max_evidence_count = (
    entity_associations[
        "evidence_count"
    ].max()
)

max_evidence_associations = (
    entity_associations[
        entity_associations[
            "evidence_count"
        ]
        ==
        max_evidence_count
    ]
)

print(
    max_evidence_associations.to_string(
        index=False
    )
)

for _, association in (
    max_evidence_associations.iterrows()
):

    entity_id_a = association[
        "entity_id_a"
    ]

    entity_id_b = association[
        "entity_id_b"
    ]

    print(
        "\n"
        "=========================================="
    )

    print(
        "ENTITIES"
    )

    print(
        "=========================================="
    )

    print(
        entities[
            entities[
                "entity_id"
            ]
            .isin([
                entity_id_a,
                entity_id_b
            ])
        ].to_string(
            index=False
        )
    )

    print(
        "\nEVIDENCE"
    )

    print(
        association_evidence[
            (
                association_evidence[
                    "entity_id_a"
                ]
                ==
                entity_id_a
            )
            &
            (
                association_evidence[
                    "entity_id_b"
                ]
                ==
                entity_id_b
            )
        ]
        .sort_values(
            [
                "evidence_rule",
                "parcel_number_a",
                "parcel_number_b"
            ]
        )
        .to_string(
            index=False
        )
    )

    print(
        "\nOCCURRENCES"
    )

    print(
        entity_occurrences[
            entity_occurrences[
                "entity_id"
            ]
            .isin([
                entity_id_a,
                entity_id_b
            ])
        ]
        .sort_values(
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        )
        [
            [
                "parcel_number",
                "entity_id",
                "entity_name",
                "role",
                "associated_address",
                "source_entity_name",
                "source_column"
            ]
        ]
        .to_string(
            index=False
        )
    )

 association_id  entity_id_a  entity_id_b association_confidence  evidence_count  evidence_rule_count
            173         1719         4342                   High              20                    5

ENTITIES
 entity_id         canonical_name  entity_type               canonical_address organization_type organization_classification_confidence  occurrence_count
      1719      GRAHAM, WILLIAM D       Person PO BOX 366, GALESBURG, MI 49053               NaN                                    NaN                 6
      4342 THE COPPER CREEK TRUST Organization PO BOX 366, GALESBURG, MI 49053             Trust                                 Medium                21

EVIDENCE
 evidence_id  entity_id_a  entity_id_b                            evidence_rule evidence_strength parcel_number_a parcel_number_b role_a      role_b                   shared_address address_type
         171         1719         4342                     SAME_PARCEL_CO_OWNER              High   07-24-201-591   07-

In [ ]:
print(
    entities[
        entities["canonical_name"].str.contains(
            r"^(?:C\s*/\s*O\s*:?|ATTN\s*:?|%)",
            regex=True,
            na=False
        )
    ][
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "canonical_address"
        ]
    ].to_string(index=False)
)

In [ ]:
print("Source occurrences:", entity_occurrences["source_occurrence_id"].nunique())
print("Entity occurrences:", len(entity_occurrences))
print("Resolved entities:", len(entities))
print("Parcel-entity-role rows:", len(parcel_entity_roles))
print("Association evidence rows:", len(association_evidence))
print("Entity associations:", len(entity_associations))

In [ ]:
# ==========================================================
# FINAL STAGE 3 ENTITY-RESOLUTION AUDIT
#
# Goals:
#   1. Find exact duplicate Organization names represented
#      by multiple entity IDs.
#
#   2. Find Organization names that differ only by
#      punctuation / spacing.
#
#   3. Find suspicious canonical names that may contain
#      source-formatting artifacts.
#
# IMPORTANT:
# These are AUDIT CANDIDATES ONLY.
# Nothing in this code merges or modifies entities.
# ==========================================================

from pathlib import Path
import re

import pandas as pd


# ==========================================================
# OUTPUT DIRECTORY
# ==========================================================

AUDIT_DIR = Path(
    "audits/entity_resolution_final"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================
# HELPERS
# ==========================================================

def _join_unique(values):
    """
    Return sorted distinct nonblank values joined by ' | '.
    """

    cleaned = sorted(
        {
            str(value).strip()
            for value in values
            if pd.notna(value)
            and str(value).strip()
        }
    )

    return " | ".join(
        cleaned
    )


def _organization_punctuation_key(name):
    """
    Conservative audit key.

    Removes punctuation differences while retaining word
    boundaries.

    Examples:

        ABSOLUTE HEAVEN LLC
        ABSOLUTE HEAVEN, LLC

    both become:

        ABSOLUTE HEAVEN LLC

    This is ONLY for candidate generation.
    """

    if pd.isna(name):
        return ""

    name = str(
        name
    ).upper().strip()

    # Periods and commas are especially common differences
    # in organization legal names.
    name = re.sub(
        r"[.,]",
        "",
        name
    )

    # Normalize apostrophe variants.
    name = name.replace(
        "’",
        "'"
    )

    # Normalize whitespace.
    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()

    return name


def _organization_compact_key(name):
    """
    More aggressive AUDIT-ONLY key.

    Removes spaces and punctuation entirely so that names
    such as:

        THE COPPER CREEK TRUST
        THE COPPERCREEK TRUST

    become comparable.

    Because this is aggressive, matches from this key must
    NEVER be automatically merged.
    """

    if pd.isna(name):
        return ""

    name = str(
        name
    ).upper().strip()

    return re.sub(
        r"[^A-Z0-9]",
        "",
        name
    )


def _suspicious_name_reason(name):
    """
    Identify canonical names that may contain source
    formatting artifacts.

    Returns a pipe-delimited reason string.
    """

    if pd.isna(name):
        return ""

    name = str(
        name
    ).strip()

    upper_name = name.upper()

    reasons = []

    if name.startswith("%"):
        reasons.append(
            "LEADING_PERCENT"
        )

    if name.startswith("&"):
        reasons.append(
            "LEADING_AMPERSAND"
        )

    if name.endswith("&"):
        reasons.append(
            "TRAILING_AMPERSAND"
        )

    if re.match(
        r"^\(\s*FKA\b",
        upper_name
    ):
        reasons.append(
            "FKA_WRAPPER"
        )

    if re.match(
        r"^FKA\b",
        upper_name
    ):
        reasons.append(
            "LEADING_FKA"
        )

    if re.match(
        r"^C/O\b",
        upper_name
    ):
        reasons.append(
            "LEADING_CARE_OF"
        )

    if re.match(
        r"^ATTN\b",
        upper_name
    ):
        reasons.append(
            "LEADING_ATTN"
        )

    return " | ".join(
        reasons
    )


# ==========================================================
# ENTITY ROLE PROFILE
#
# Gives each entity useful context:
#   - number of parcels
#   - number of Owner parcels
#   - roles observed
# ==========================================================

role_profile = (
    parcel_entity_roles
    .groupby(
        "entity_id",
        as_index=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),

        roles=(
            "role",
            _join_unique
        )
    )
)


owner_profile = (
    parcel_entity_roles[
        parcel_entity_roles[
            "role"
        ]
        == "Owner"
    ]
    .groupby(
        "entity_id",
        as_index=False
    )
    .agg(
        owner_parcel_count=(
            "parcel_number",
            "nunique"
        )
    )
)


role_profile = (
    role_profile
    .merge(
        owner_profile,
        on="entity_id",
        how="left"
    )
)

role_profile[
    "owner_parcel_count"
] = (
    role_profile[
        "owner_parcel_count"
    ]
    .fillna(0)
    .astype(int)
)


# ==========================================================
# OCCURRENCE / SOURCE PROFILE
#
# Useful for seeing how an entity actually appeared in the
# original columns.
# ==========================================================

source_profile = (
    entity_occurrences
    .groupby(
        "entity_id",
        as_index=False
    )
    .agg(
        source_names=(
            "source_entity_name",
            _join_unique
        ),

        source_columns=(
            "source_column",
            _join_unique
        )
    )
)


# ==========================================================
# BASE ORGANIZATION AUDIT TABLE
# ==========================================================

organizations = (
    entities[
        entities[
            "entity_type"
        ]
        == "Organization"
    ]
    .copy()
)

organizations = (
    organizations
    .merge(
        role_profile,
        on="entity_id",
        how="left"
    )
    .merge(
        source_profile,
        on="entity_id",
        how="left"
    )
)

organizations[
    "punctuation_key"
] = (
    organizations[
        "canonical_name"
    ]
    .map(
        _organization_punctuation_key
    )
)

organizations[
    "compact_key"
] = (
    organizations[
        "canonical_name"
    ]
    .map(
        _organization_compact_key
    )
)


# ==========================================================
# AUDIT 1
# EXACT SAME ORGANIZATION NAME, MULTIPLE ENTITY IDs
# ==========================================================

exact_duplicate_names = (
    organizations
    .groupby(
        "canonical_name"
    )[
        "entity_id"
    ]
    .nunique()
)

exact_duplicate_names = set(
    exact_duplicate_names[
        exact_duplicate_names > 1
    ].index
)


exact_duplicate_org_detail = (
    organizations[
        organizations[
            "canonical_name"
        ].isin(
            exact_duplicate_names
        )
    ]
    .copy()
    .sort_values(
        [
            "canonical_name",
            "canonical_address",
            "entity_id"
        ],
        na_position="first"
    )
)


exact_duplicate_org_summary = (
    exact_duplicate_org_detail
    .groupby(
        "canonical_name",
        as_index=False
    )
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),

        entity_ids=(
            "entity_id",
            lambda x:
                " | ".join(
                    map(
                        str,
                        sorted(
                            set(x)
                        )
                    )
                )
        ),

        canonical_addresses=(
            "canonical_address",
            _join_unique
        ),

        total_occurrences=(
            "occurrence_count",
            "sum"
        ),

        total_parcels=(
            "parcel_count",
            "sum"
        ),

        total_owner_parcels=(
            "owner_parcel_count",
            "sum"
        ),

        source_names=(
            "source_names",
            _join_unique
        )
    )
    .sort_values(
        [
            "entity_count",
            "canonical_name"
        ],
        ascending=[
            False,
            True
        ]
    )
)


# ==========================================================
# AUDIT 2A
# PUNCTUATION-ONLY ORGANIZATION VARIANTS
#
# Example:
#   ABSOLUTE HEAVEN LLC
#   ABSOLUTE HEAVEN, LLC
# ==========================================================

punctuation_group_stats = (
    organizations
    .groupby(
        "punctuation_key"
    )
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),

        distinct_name_count=(
            "canonical_name",
            "nunique"
        )
    )
    .reset_index()
)


punctuation_candidate_keys = set(
    punctuation_group_stats.loc[
        (
            punctuation_group_stats[
                "entity_count"
            ]
            > 1
        )
        &
        (
            punctuation_group_stats[
                "distinct_name_count"
            ]
            > 1
        ),
        "punctuation_key"
    ]
)


punctuation_variant_detail = (
    organizations[
        organizations[
            "punctuation_key"
        ].isin(
            punctuation_candidate_keys
        )
    ]
    .copy()
)


punctuation_variant_detail[
    "audit_match_type"
] = (
    "PUNCTUATION_VARIANT"
)


# ==========================================================
# AUDIT 2B
# COMPACT/SPACING ORGANIZATION VARIANTS
#
# Example:
#   THE COPPER CREEK TRUST
#   THE COPPERCREEK TRUST
#
# This is intentionally broader and therefore needs more
# manual scrutiny.
# ==========================================================

compact_group_stats = (
    organizations
    .groupby(
        "compact_key"
    )
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),

        distinct_name_count=(
            "canonical_name",
            "nunique"
        )
    )
    .reset_index()
)


compact_candidate_keys = set(
    compact_group_stats.loc[
        (
            compact_group_stats[
                "entity_count"
            ]
            > 1
        )
        &
        (
            compact_group_stats[
                "distinct_name_count"
            ]
            > 1
        ),
        "compact_key"
    ]
)


compact_variant_detail = (
    organizations[
        organizations[
            "compact_key"
        ].isin(
            compact_candidate_keys
        )
    ]
    .copy()
)


# Remove groups already captured by the more conservative
# punctuation audit.
compact_variant_detail = (
    compact_variant_detail[
        ~compact_variant_detail[
            "punctuation_key"
        ].isin(
            punctuation_candidate_keys
        )
    ]
    .copy()
)


compact_variant_detail[
    "audit_match_type"
] = (
    "COMPACT_SPACING_VARIANT"
)


# ==========================================================
# COMBINE ORGANIZATION VARIANT AUDITS
# ==========================================================

organization_variant_detail = pd.concat(
    [
        punctuation_variant_detail,
        compact_variant_detail
    ],
    ignore_index=True
)


if not organization_variant_detail.empty:

    organization_variant_detail[
        "audit_group_key"
    ] = (
        organization_variant_detail[
            "audit_match_type"
        ].astype(str)
        +
        "::"
        +
        organization_variant_detail.apply(
            lambda row:
                (
                    row[
                        "punctuation_key"
                    ]
                    if row[
                        "audit_match_type"
                    ]
                    == "PUNCTUATION_VARIANT"
                    else row[
                        "compact_key"
                    ]
                ),
            axis=1
        )
    )

    organization_variant_detail = (
        organization_variant_detail
        .sort_values(
            [
                "audit_match_type",
                "audit_group_key",
                "canonical_name",
                "canonical_address",
                "entity_id"
            ],
            na_position="first"
        )
    )


    organization_variant_summary = (
        organization_variant_detail
        .groupby(
            [
                "audit_match_type",
                "audit_group_key"
            ],
            as_index=False
        )
        .agg(
            entity_count=(
                "entity_id",
                "nunique"
            ),

            distinct_name_count=(
                "canonical_name",
                "nunique"
            ),

            canonical_names=(
                "canonical_name",
                _join_unique
            ),

            entity_ids=(
                "entity_id",
                lambda x:
                    " | ".join(
                        map(
                            str,
                            sorted(
                                set(x)
                            )
                        )
                    )
            ),

            canonical_addresses=(
                "canonical_address",
                _join_unique
            ),

            total_occurrences=(
                "occurrence_count",
                "sum"
            ),

            source_names=(
                "source_names",
                _join_unique
            )
        )
        .sort_values(
            [
                "audit_match_type",
                "entity_count",
                "canonical_names"
            ],
            ascending=[
                True,
                False,
                True
            ]
        )
    )

else:

    organization_variant_summary = (
        pd.DataFrame()
    )


# ==========================================================
# AUDIT 3
# SUSPICIOUS CANONICAL NAMES
#
# Run across ALL entity types, not just organizations.
# ==========================================================

suspicious_entities = (
    entities
    .copy()
)

suspicious_entities[
    "suspicious_name_reason"
] = (
    suspicious_entities[
        "canonical_name"
    ]
    .map(
        _suspicious_name_reason
    )
)


suspicious_entities = (
    suspicious_entities[
        suspicious_entities[
            "suspicious_name_reason"
        ]
        != ""
    ]
    .copy()
)


suspicious_entity_summary = (
    suspicious_entities
    .merge(
        role_profile,
        on="entity_id",
        how="left"
    )
    .merge(
        source_profile,
        on="entity_id",
        how="left"
    )
    .sort_values(
        [
            "suspicious_name_reason",
            "canonical_name",
            "entity_id"
        ]
    )
)


suspicious_entity_ids = set(
    suspicious_entity_summary[
        "entity_id"
    ]
)


# Full occurrence-level lineage for suspicious entities.
suspicious_entity_occurrences = (
    entity_occurrences[
        entity_occurrences[
            "entity_id"
        ].isin(
            suspicious_entity_ids
        )
    ]
    .copy()
    .merge(
        suspicious_entity_summary[
            [
                "entity_id",
                "canonical_name",
                "suspicious_name_reason"
            ]
        ],
        on="entity_id",
        how="left"
    )
    .sort_values(
        [
            "suspicious_name_reason",
            "canonical_name",
            "entity_id",
            "parcel_number",
            "occurrence_id"
        ]
    )
)


# ==========================================================
# EXPORT AUDIT FILES
# ==========================================================

exact_duplicate_org_summary.to_csv(
    AUDIT_DIR
    /
    "exact_duplicate_org_summary.csv",
    index=False
)

exact_duplicate_org_detail.to_csv(
    AUDIT_DIR
    /
    "exact_duplicate_org_detail.csv",
    index=False
)

organization_variant_summary.to_csv(
    AUDIT_DIR
    /
    "organization_name_variant_summary.csv",
    index=False
)

organization_variant_detail.to_csv(
    AUDIT_DIR
    /
    "organization_name_variant_detail.csv",
    index=False
)

suspicious_entity_summary.to_csv(
    AUDIT_DIR
    /
    "suspicious_entity_summary.csv",
    index=False
)

suspicious_entity_occurrences.to_csv(
    AUDIT_DIR
    /
    "suspicious_entity_occurrences.csv",
    index=False
)


# ==========================================================
# CONSOLE SUMMARY
# ==========================================================

print(
    "\n"
    "==========================================\n"
    "FINAL ENTITY-RESOLUTION AUDIT\n"
    "=========================================="
)

print(
    "\nExact Organization names represented by "
    "multiple entity IDs:",
    len(
        exact_duplicate_org_summary
    )
)

print(
    "Entities involved in exact-name duplicates:",
    len(
        exact_duplicate_org_detail
    )
)

print(
    "\nOrganization variant candidate groups:",
    len(
        organization_variant_summary
    )
)

if not organization_variant_summary.empty:

    print(
        "\nVariant groups by match type:"
    )

    print(
        organization_variant_summary[
            "audit_match_type"
        ]
        .value_counts()
    )


print(
    "\nSuspicious canonical entities:",
    len(
        suspicious_entity_summary
    )
)

if not suspicious_entity_summary.empty:

    print(
        "\nSuspicious-name reasons:"
    )

    print(
        suspicious_entity_summary[
            "suspicious_name_reason"
        ]
        .value_counts()
    )


print(
    "\nAudit files written to:"
)

print(
    AUDIT_DIR.resolve()
)

In [ ]:
# ==========================================================
# FINAL STAGE 3 — DECISION AUDIT
#
# Purpose:
#   A. Determine which exact-name Organization duplicates
#      occur on the same parcel.
#
#   B. Determine which punctuation/spacing variants occur
#      on the same parcel.
#
#   C. Check whether routing-prefixed entities (C/O, ATTN,
#      %) have corresponding unprefixed entities.
#
#   D. Show full source-row context for ampersand/FKA cases.
#
# NO DATA IS MODIFIED.
# ==========================================================

from pathlib import Path
import re
import pandas as pd


AUDIT_DIR = Path(
    "audits/entity_resolution_final_decision"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================
# HELPERS
# ==========================================================

def join_unique(values):

    values = sorted(
        {
            str(value).strip()
            for value in values
            if pd.notna(value)
            and str(value).strip()
        }
    )

    return " | ".join(values)


def punctuation_key(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    # Remove punctuation differences commonly seen in
    # organization legal names.
    name = re.sub(
        r"[.,]",
        "",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


def compact_key(name):

    if pd.isna(name):
        return ""

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(name).upper()
    )


def strip_routing_prefix(name):
    """
    Audit-only representation of the underlying identity.

    Does NOT alter entity_occurrences or source names.
    """

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    # C/O, C/O:, C / O, etc.
    name = re.sub(
        r"^C\s*/\s*O\s*:?\s*",
        "",
        name
    )

    # ATTN / ATTN:
    name = re.sub(
        r"^ATTN\s*:?\s*",
        "",
        name
    )

    # Percent sign used as routing/care-of notation.
    name = re.sub(
        r"^%\s*",
        "",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


# ==========================================================
# ENTITY LOOKUP
# ==========================================================

entity_lookup = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .copy()
)


# ==========================================================
# A. EXACT-NAME ORGANIZATION DUPLICATES ON SAME PARCEL
# ==========================================================

org_occurrences = (
    entity_occurrences[
        entity_occurrences[
            "entity_type"
        ]
        == "Organization"
    ]
    .merge(
        entity_lookup[
            [
                "entity_id",
                "canonical_name",
                "canonical_address"
            ]
        ],
        on="entity_id",
        how="left"
    )
)


exact_same_parcel_candidates = (
    org_occurrences
    .groupby(
        [
            "parcel_number",
            "canonical_name"
        ],
        as_index=False
    )
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),

        entity_ids=(
            "entity_id",
            lambda x:
                " | ".join(
                    map(
                        str,
                        sorted(
                            set(x)
                        )
                    )
                )
        ),

        roles=(
            "role",
            join_unique
        ),

        source_columns=(
            "source_column",
            join_unique
        ),

        source_names=(
            "source_entity_name",
            join_unique
        ),

        addresses=(
            "associated_address",
            join_unique
        )
    )
)

exact_same_parcel_candidates = (
    exact_same_parcel_candidates[
        exact_same_parcel_candidates[
            "entity_count"
        ]
        > 1
    ]
    .copy()
    .sort_values(
        [
            "canonical_name",
            "parcel_number"
        ]
    )
)


# ==========================================================
# B. ORGANIZATION VARIANTS ON SAME PARCEL
# ==========================================================

variant_occurrences = (
    org_occurrences.copy()
)

variant_occurrences[
    "punctuation_key"
] = (
    variant_occurrences[
        "canonical_name"
    ]
    .map(
        punctuation_key
    )
)

variant_occurrences[
    "compact_key"
] = (
    variant_occurrences[
        "canonical_name"
    ]
    .map(
        compact_key
    )
)


punctuation_same_parcel = (
    variant_occurrences
    .groupby(
        [
            "parcel_number",
            "punctuation_key"
        ],
        as_index=False
    )
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),

        distinct_name_count=(
            "canonical_name",
            "nunique"
        ),

        entity_ids=(
            "entity_id",
            lambda x:
                " | ".join(
                    map(
                        str,
                        sorted(
                            set(x)
                        )
                    )
                )
        ),

        canonical_names=(
            "canonical_name",
            join_unique
        ),

        roles=(
            "role",
            join_unique
        ),

        addresses=(
            "associated_address",
            join_unique
        ),

        source_names=(
            "source_entity_name",
            join_unique
        )
    )
)

punctuation_same_parcel = (
    punctuation_same_parcel[
        (
            punctuation_same_parcel[
                "entity_count"
            ]
            > 1
        )
        &
        (
            punctuation_same_parcel[
                "distinct_name_count"
            ]
            > 1
        )
    ]
    .copy()
)


compact_same_parcel = (
    variant_occurrences
    .groupby(
        [
            "parcel_number",
            "compact_key"
        ],
        as_index=False
    )
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),

        distinct_name_count=(
            "canonical_name",
            "nunique"
        ),

        entity_ids=(
            "entity_id",
            lambda x:
                " | ".join(
                    map(
                        str,
                        sorted(
                            set(x)
                        )
                    )
                )
        ),

        canonical_names=(
            "canonical_name",
            join_unique
        ),

        roles=(
            "role",
            join_unique
        ),

        addresses=(
            "associated_address",
            join_unique
        ),

        source_names=(
            "source_entity_name",
            join_unique
        )
    )
)

compact_same_parcel = (
    compact_same_parcel[
        (
            compact_same_parcel[
                "entity_count"
            ]
            > 1
        )
        &
        (
            compact_same_parcel[
                "distinct_name_count"
            ]
            > 1
        )
    ]
    .copy()
)


# ==========================================================
# C. ROUTING-PREFIX MATCH AUDIT
# ==========================================================

routing_pattern = (
    r"^(?:"
    r"C\s*/\s*O\s*:?"
    r"|ATTN\s*:?"
    r"|%"
    r")"
)

routing_entities = (
    entities[
        entities[
            "canonical_name"
        ]
        .str.contains(
            routing_pattern,
            regex=True,
            na=False
        )
    ]
    .copy()
)

routing_entities[
    "stripped_name"
] = (
    routing_entities[
        "canonical_name"
    ]
    .map(
        strip_routing_prefix
    )
)


all_entities_for_match = (
    entities.copy()
)

all_entities_for_match[
    "normalized_name"
] = (
    all_entities_for_match[
        "canonical_name"
    ]
    .map(
        lambda x:
            punctuation_key(
                strip_routing_prefix(x)
            )
    )
)


routing_entities[
    "normalized_stripped_name"
] = (
    routing_entities[
        "stripped_name"
    ]
    .map(
        punctuation_key
    )
)


routing_prefix_matches = (
    routing_entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "canonical_address",
            "occurrence_count",
            "stripped_name",
            "normalized_stripped_name"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "routing_entity_id",

            "canonical_name":
                "routing_name",

            "entity_type":
                "routing_entity_type",

            "canonical_address":
                "routing_address",

            "occurrence_count":
                "routing_occurrence_count"
        }
    )
    .merge(
        all_entities_for_match[
            [
                "entity_id",
                "canonical_name",
                "entity_type",
                "canonical_address",
                "occurrence_count",
                "normalized_name"
            ]
        ],
        left_on="normalized_stripped_name",
        right_on="normalized_name",
        how="left"
    )
)


# Remove the routing entity matching itself.
routing_prefix_matches = (
    routing_prefix_matches[
        routing_prefix_matches[
            "routing_entity_id"
        ]
        !=
        routing_prefix_matches[
            "entity_id"
        ]
    ]
    .copy()
)


# Useful indicator: exact same address.
routing_prefix_matches[
    "same_address"
] = (
    routing_prefix_matches[
        "routing_address"
    ].fillna("")
    ==
    routing_prefix_matches[
        "canonical_address"
    ].fillna("")
)


# ==========================================================
# D. FULL SOURCE CONTEXT FOR AMPERSAND / FKA CASES
# ==========================================================

suspicious_source_names = (
    entity_occurrences[
        entity_occurrences[
            "entity_name"
        ]
        .str.contains(
            (
                r"(?:^&)|(?:&$)|(?:^\(\s*FKA\b)"
            ),
            regex=True,
            na=False
        )
    ][
        [
            "parcel_number",
            "entity_id",
            "entity_name",
            "source_entity_name",
            "source_column",
            "role"
        ]
    ]
    .copy()
)


suspicious_parcels = set(
    suspicious_source_names[
        "parcel_number"
    ]
)


source_columns = [
    "parcel_number",
    "owner_name",
    "owner_name_2",
    "owner_care_of",
    "owner_address",
    "taxpayer_name",
    "taxpayer_care_of",
    "taxpayer_address",
    "tax_send_to_name"
]


ampersand_fka_source_context = (
    clean_df[
        clean_df[
            "parcel_number"
        ].isin(
            suspicious_parcels
        )
    ][
        source_columns
    ]
    .copy()
)


ampersand_fka_source_context = (
    ampersand_fka_source_context
    .merge(
        suspicious_source_names,
        on="parcel_number",
        how="left"
    )
    .sort_values(
        [
            "parcel_number",
            "source_column"
        ]
    )
)


# ==========================================================
# EXPORT
# ==========================================================

exact_same_parcel_candidates.to_csv(
    AUDIT_DIR
    /
    "exact_org_same_parcel_candidates.csv",
    index=False
)

punctuation_same_parcel.to_csv(
    AUDIT_DIR
    /
    "punctuation_variant_same_parcel.csv",
    index=False
)

compact_same_parcel.to_csv(
    AUDIT_DIR
    /
    "compact_variant_same_parcel.csv",
    index=False
)

routing_prefix_matches.to_csv(
    AUDIT_DIR
    /
    "routing_prefix_matches.csv",
    index=False
)

ampersand_fka_source_context.to_csv(
    AUDIT_DIR
    /
    "ampersand_fka_source_context.csv",
    index=False
)


# ==========================================================
# CONSOLE SUMMARY
# ==========================================================

print(
    "\n"
    "==========================================\n"
    "FINAL STAGE 3 DECISION AUDIT\n"
    "=========================================="
)

print(
    "\nExact-name Organization duplicates "
    "sharing a parcel:",
    len(
        exact_same_parcel_candidates
    )
)

print(
    "Punctuation-variant Organization groups "
    "sharing a parcel:",
    len(
        punctuation_same_parcel
    )
)

print(
    "Compact/spacing Organization groups "
    "sharing a parcel:",
    len(
        compact_same_parcel
    )
)

print(
    "\nRouting-prefixed entities:",
    len(
        routing_entities
    )
)

print(
    "Routing-prefix candidate matches:",
    len(
        routing_prefix_matches
    )
)

if not routing_prefix_matches.empty:

    print(
        "Routing-prefix matches with exact "
        "same address:",
        routing_prefix_matches[
            "same_address"
        ].sum()
    )

print(
    "\nAmpersand/FKA source-context rows:",
    len(
        ampersand_fka_source_context
    )
)

print(
    "\nAudit files written to:"
)

print(
    AUDIT_DIR.resolve()
)

In [ ]:
# ==========================================================
# OWNER_NAME / OWNER_NAME_2 CONTINUATION-LINE AUDIT
# ==========================================================

def owner_line_audit_key(value):
    """
    Create a comparison-only representation for detecting
    possible source-line continuations.

    This does not modify production entity names.
    """

    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    value = re.sub(
        r"[^A-Z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [ ]:
name_columns = [
    "owner_name",
    "owner_name_2",
    "owner_care_of",
    "taxpayer_name",
    "taxpayer_care_of",
    "tax_send_to_name"
]

known_source_name_keys = set()

for column in name_columns:

    if column not in clean_df.columns:
        continue

    known_source_name_keys.update(
        clean_df[column]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[
            lambda x:
                x != ""
        ]
        .map(
            owner_line_audit_key
        )
    )

known_source_name_keys.discard("")

In [ ]:
owner_line_pairs = (
    clean_df[
        (
            clean_df[
                "owner_name"
            ].notna()
        )
        &
        (
            clean_df[
                "owner_name"
            ].str.strip()
            != ""
        )
        &
        (
            clean_df[
                "owner_name_2"
            ].notna()
        )
        &
        (
            clean_df[
                "owner_name_2"
            ].str.strip()
            != ""
        )
    ]
    [
        [
            "parcel_number",
            "owner_name",
            "owner_name_2",
            "owner_address"
        ]
    ]
    .drop_duplicates()
    .copy()
)

In [ ]:
owner_line_pairs[
    "forward_combined_name"
] = (
    owner_line_pairs[
        "owner_name"
    ].str.strip()
    +
    " "
    +
    owner_line_pairs[
        "owner_name_2"
    ].str.strip()
)

owner_line_pairs[
    "reverse_combined_name"
] = (
    owner_line_pairs[
        "owner_name_2"
    ].str.strip()
    +
    " "
    +
    owner_line_pairs[
        "owner_name"
    ].str.strip()
)

owner_line_pairs[
    "forward_key"
] = (
    owner_line_pairs[
        "forward_combined_name"
    ]
    .map(
        owner_line_audit_key
    )
)

owner_line_pairs[
    "reverse_key"
] = (
    owner_line_pairs[
        "reverse_combined_name"
    ]
    .map(
        owner_line_audit_key
    )
)

In [ ]:
generic_continuation_pattern = re.compile(
    r"""
    ^
    (?:
        TRUST
        |
        LIVING\s+TRUST
        |
        REVOCABLE\s+TRUST
        |
        REVOCABLE\s+TRST
        |
        LIVING\s+TRST
        |
        ESTATE\s+OF
    )
    $
    """,
    flags=re.IGNORECASE | re.VERBOSE
)


def continuation_reasons(row):

    reasons = []

    owner_name = str(
        row["owner_name"]
    ).strip()

    owner_name_2 = str(
        row["owner_name_2"]
    ).strip()

    # Does combining the two lines reproduce a complete
    # source name that occurs elsewhere in the data?
    if (
        row["forward_key"]
        in known_source_name_keys
    ):
        reasons.append(
            "FORWARD_COMBINATION_EXISTS_ELSEWHERE"
        )

    if (
        row["reverse_key"]
        in known_source_name_keys
    ):
        reasons.append(
            "REVERSE_COMBINATION_EXISTS_ELSEWHERE"
        )

    # Examples:
    # KALAMAZOO EAST CONGREGATION OF
    # JEHOVAH'S WITNESS
    if re.search(
        r"\bOF$",
        owner_name,
        flags=re.IGNORECASE
    ):
        reasons.append(
            "OWNER_NAME_ENDS_WITH_OF"
        )

    # Examples:
    # DECKER, LYDIA
    # LIVING TRUST
    #
    # APPLEGATE, NEIL E & SHARON K
    # REVOCABLE TRUST
    if generic_continuation_pattern.fullmatch(
        owner_name_2
    ):
        reasons.append(
            "OWNER_NAME_2_GENERIC_CONTINUATION"
        )

    # Example:
    # WILLIAMS BRIAN R (DECEASED)
    # ESTATE OF
    if generic_continuation_pattern.fullmatch(
        owner_name
    ):
        reasons.append(
            "OWNER_NAME_GENERIC_CONTINUATION"
        )

    # Example:
    # STAUFFER JENNIFER
    # TRUSTEE OF THE BELL FAMILY TRUST
    if re.match(
        r"^TRUSTEE\s+OF\b",
        owner_name_2,
        flags=re.IGNORECASE
    ):
        reasons.append(
            "OWNER_NAME_2_STARTS_TRUSTEE_OF"
        )

    return tuple(reasons)

In [ ]:
owner_line_pairs[
    "continuation_reasons"
] = (
    owner_line_pairs
    .apply(
        continuation_reasons,
        axis=1
    )
)

owner_line_continuation_candidates = (
    owner_line_pairs[
        owner_line_pairs[
            "continuation_reasons"
        ].map(bool)
    ]
    .copy()
    .sort_values(
        [
            "owner_name",
            "owner_name_2",
            "parcel_number"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

print(
    "Owner-name pairs with both lines populated:",
    len(owner_line_pairs)
)

print(
    "Potential continuation-line pairs:",
    len(
        owner_line_continuation_candidates
    )
)

owner_line_continuation_candidates

In [ ]:
continuation_reason_counts = (
    owner_line_continuation_candidates[
        "continuation_reasons"
    ]
    .explode()
    .value_counts()
)

continuation_reason_counts

In [ ]:
owner_line_continuation_candidates.to_csv(
    "owner_line_continuation_candidates.csv",
    index=False
)

## Repeated surname audit

In [ ]:
def has_repeated_surname_comma_structure(name):
    """
    Flag names that appear to contain two full person names
    separated by a comma, where both sides begin with the
    same surname.

    Example
    -------
    DARDEN MURPHY JR, DARDEN VICENT B
        -> True

    SMITH, JOHN A
        -> False
    """

    if pd.isna(name):
        return False

    name = str(name).upper().strip()

    # This audit is specifically for a two-part comma structure.
    parts = [
        part.strip()
        for part in name.split(",")
        if part.strip()
    ]

    if len(parts) != 2:
        return False

    left_tokens = re.findall(
        r"[A-Z]+(?:['-][A-Z]+)*",
        parts[0]
    )

    right_tokens = re.findall(
        r"[A-Z]+(?:['-][A-Z]+)*",
        parts[1]
    )

    # Each side should look substantial enough to represent
    # a surname plus at least one given-name token.
    if (
            len(left_tokens) < 2
            or len(right_tokens) < 2
    ):
        return False

    return left_tokens[0] == right_tokens[0]

In [ ]:
repeating_surname_comma_audit = (
    entity_occurrences[
        (
            entity_occurrences["entity_type"]
            == "Person"
        )
        &
        (
            entity_occurrences["entity_name"]
            .apply(
                has_repeated_surname_comma_structure
            )
        )
    ]
    [
        [
            "entity_id",
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "parcel_number",
            "role",
            "associated_address",
            "source_column"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "entity_name",
            "parcel_number",
            "role"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Person occurrences with a repeated-surname "
    "comma structure:",
    len(repeating_surname_comma_audit)
)

print(
    "Distinct entity names:",
    repeating_surname_comma_audit[
        "entity_name"
    ].nunique()
)

repeating_surname_comma_audit

In [ ]:
repeating_surname_comma_summary = (
    repeating_surname_comma_audit
    .groupby(
        [
            "entity_id",
            "entity_name",
            "associated_address"
        ],
        dropna=False
    )
    .agg(
        occurrence_count=(
            "source_occurrence_id",
            "nunique"
        ),
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        ),
        roles=(
            "role",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        ),
        source_names=(
            "source_entity_name",
            lambda x:
                " | ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
    .sort_values(
        "entity_name"
    )
    .reset_index(drop=True)
)

print(
    "Distinct suspicious entities:",
    len(repeating_surname_comma_summary)
)

repeating_surname_comma_summary

In [ ]:
darden_joint_check = (
    entity_occurrences[
        entity_occurrences[
            "source_entity_name"
        ].isin(
            [
                "DARDEN MURPHY JR, DARDEN VICENT B",
                "DARDEN RICKIE L, DARDEN VANASSA A"
            ]
        )
    ]
    [
        [
            "source_entity_name",
            "entity_name",
            "entity_type",
            "parcel_number",
            "role",
            "associated_address"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "source_entity_name",
            "role"
        ]
    )
    .reset_index(drop=True)
)

darden_joint_check

In [ ]:
print(
    "Occurrence count:",
    entities["occurrence_count"].sum()
)

In [ ]:
repeating_surname_comma_audit = (
    entity_occurrences[
        (
            entity_occurrences["entity_type"]
            == "Person"
        )
        &
        (
            entity_occurrences["entity_name"]
            .apply(
                has_repeated_surname_comma_structure
            )
        )
    ]
)

print(
    "Remaining repeated-surname comma Persons:",
    len(repeating_surname_comma_audit)
)

## Association evidence audit

In [ ]:
print(
    "Association evidence rows:",
    len(association_evidence)
)

association_evidence[
    "evidence_strength"
].value_counts()

In [ ]:
association_evidence[
    "evidence_rule"
].value_counts()

In [ ]:
print(
    "Duplicate evidence rows:",
    association_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "parcel_number_a",
            "parcel_number_b",
            "role_a",
            "role_b"
        ]
    ).sum()
)

In [ ]:
print(
    "Total association evidence rows:",
    len(association_evidence)
)

print("\nEvidence rules:")
print(
    association_evidence[
        "evidence_rule"
    ].value_counts()
)

print("\nEvidence strengths:")
print(
    association_evidence[
        "evidence_strength"
    ].value_counts()
)

In [ ]:
shared_owner_evidence = (
    association_evidence[
        association_evidence[
            "evidence_rule"
        ]
        == "SHARED_OWNER_ADDRESS"
    ]
)

print(
    "Shared-owner-address evidence rows:",
    len(shared_owner_evidence)
)

print(
    "Distinct entity pairs:",
    shared_owner_evidence[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared addresses:",
    shared_owner_evidence[
        "shared_address"
    ].nunique()
)

print(
    "Duplicate shared-owner evidence:",
    shared_owner_evidence.duplicated(
        subset=[
            "entity_id_a",
            "entity_id_b",
            "evidence_rule",
            "shared_address"
        ]
    ).sum()
)

In [ ]:
print(
    "Shared-owner rows with parcel_number_a populated:",
    shared_owner_evidence[
        "parcel_number_a"
    ].notna().sum()
)

print(
    "Shared-owner rows with parcel_number_b populated:",
    shared_owner_evidence[
        "parcel_number_b"
    ].notna().sum()
)

In [ ]:
# Every evidence ID should be unique.
assert association_evidence[
    "evidence_id"
].is_unique

# Entity pairs should always be normalized.
assert (
    association_evidence[
        "entity_id_a"
    ]
    <
    association_evidence[
        "entity_id_b"
    ]
).all()

# Every referenced entity should actually exist.
valid_entity_ids = set(
    entities["entity_id"]
)

assert set(
    association_evidence["entity_id_a"]
).issubset(valid_entity_ids)

assert set(
    association_evidence["entity_id_b"]
).issubset(valid_entity_ids)

# Shared-owner-address evidence should have exactly
# the expected characteristics.
shared_owner = association_evidence[
    association_evidence["evidence_rule"]
    == "SHARED_OWNER_ADDRESS"
]

assert (
    shared_owner["evidence_strength"]
    == "Medium"
).all()

assert (
    shared_owner["address_type"]
    == "Owner"
).all()

assert (
    shared_owner["role_a"]
    == "Owner"
).all()

assert (
    shared_owner["role_b"]
    == "Owner"
).all()

assert shared_owner[
    "shared_address"
].notna().all()

assert (
    shared_owner[
        "shared_address"
    ].str.strip()
    != ""
).all()

print("Stage 5 shared-owner-address validation passed.")

## Shared taxpayer address audit

In [ ]:
# ----------------------------------------------------------
# One taxpayer address per parcel
# ----------------------------------------------------------

parcel_taxpayer_addresses = (
    clean_df[
        [
            "parcel_number",
            "taxpayer_address"
        ]
    ]
    .copy()
)

parcel_taxpayer_addresses = (
    parcel_taxpayer_addresses[
        (
            parcel_taxpayer_addresses[
                "taxpayer_address"
            ].notna()
        )
        &
        (
            parcel_taxpayer_addresses[
                "taxpayer_address"
            ].str.strip()
            != ""
        )
    ]
    .drop_duplicates()
)

In [ ]:
taxpayer_address_counts = (
    parcel_taxpayer_addresses
    .groupby("parcel_number")[
        "taxpayer_address"
    ]
    .nunique()
)

print(
    "Parcels with more than one taxpayer address:",
    (taxpayer_address_counts > 1).sum()
)

In [ ]:
owner_entities_by_parcel = (
    parcel_entity_roles[
        parcel_entity_roles[
            "role"
        ] == "Owner"
    ]
    [
        [
            "parcel_number",
            "entity_id"
        ]
    ]
    .drop_duplicates()
)

owner_taxpayer_address_entities = (
    owner_entities_by_parcel
    .merge(
        parcel_taxpayer_addresses,
        on="parcel_number",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        entities[
            [
                "entity_id",
                "entity_type",
                "canonical_name"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
print(
    "Owner rows with missing entity type:",
    owner_taxpayer_address_entities[
        "entity_type"
    ].isna().sum()
)

### Profile taxpayer addresses

In [ ]:
taxpayer_address_profile = (
    owner_taxpayer_address_entities
    .groupby(
        "taxpayer_address"
    )
    .agg(
        distinct_entities=(
            "entity_id",
            "nunique"
        ),
        distinct_parcels=(
            "parcel_number",
            "nunique"
        )
    )
    .reset_index()
)

In [ ]:
person_counts = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Person"
    ]
    .groupby(
        "taxpayer_address"
    )[
        "entity_id"
    ]
    .nunique()
    .rename(
        "distinct_person_entities"
    )
)

organization_counts = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    .groupby(
        "taxpayer_address"
    )[
        "entity_id"
    ]
    .nunique()
    .rename(
        "distinct_organization_entities"
    )
)

taxpayer_address_profile = (
    taxpayer_address_profile
    .merge(
        person_counts,
        on="taxpayer_address",
        how="left"
    )
    .merge(
        organization_counts,
        on="taxpayer_address",
        how="left"
    )
    .fillna(
        {
            "distinct_person_entities": 0,
            "distinct_organization_entities": 0
        }
    )
)

taxpayer_address_profile[
    [
        "distinct_person_entities",
        "distinct_organization_entities"
    ]
] = (
    taxpayer_address_profile[
        [
            "distinct_person_entities",
            "distinct_organization_entities"
        ]
    ]
    .astype(int)
)

taxpayer_address_profile = (
    taxpayer_address_profile
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
print(
    "Taxpayer addresses:",
    len(taxpayer_address_profile)
)

print(
    "Addresses linked to >1 owner entity:",
    (
        taxpayer_address_profile[
            "distinct_entities"
        ] > 1
    ).sum()
)

print(
    "Addresses containing both Person and Organization owners:",
    (
        (
            taxpayer_address_profile[
                "distinct_person_entities"
            ] > 0
        )
        &
        (
            taxpayer_address_profile[
                "distinct_organization_entities"
            ] > 0
        )
    ).sum()
)

taxpayer_address_profile[
    [
        "distinct_entities",
        "distinct_parcels"
    ]
].describe()

In [ ]:
taxpayer_address_persons = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Person"
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "canonical_name",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "parcel_number":
                "person_parcel",
            "entity_id":
                "person_id",
            "canonical_name":
                "person_name"
        }
    )
)

taxpayer_address_organizations = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "canonical_name",
            "taxpayer_address"
        ]
    ]
    .rename(
        columns={
            "parcel_number":
                "organization_parcel",
            "entity_id":
                "organization_id",
            "canonical_name":
                "organization_name"
        }
    )
)

shared_taxpayer_address_candidates = (
    taxpayer_address_organizations
    .merge(
        taxpayer_address_persons,
        on="taxpayer_address",
        how="inner"
    )
)

shared_taxpayer_address_candidates = (
    shared_taxpayer_address_candidates[
        shared_taxpayer_address_candidates[
            "organization_parcel"
        ]
        !=
        shared_taxpayer_address_candidates[
            "person_parcel"
        ]
    ]
    .copy()
)

In [ ]:
print(
    "Cross-parcel shared-taxpayer-address candidate rows:",
    len(shared_taxpayer_address_candidates)
)

print(
    "Distinct Person-Organization pairs:",
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared taxpayer addresses:",
    shared_taxpayer_address_candidates[
        "taxpayer_address"
    ].nunique()
)

print(
    "Distinct Person-Organization-address combinations:",
    shared_taxpayer_address_candidates[
        [
            "person_id",
            "organization_id",
            "taxpayer_address"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
candidate_taxpayer_addresses = set(
    shared_taxpayer_address_candidates[
        "taxpayer_address"
    ]
)

taxpayer_address_member_summary = (
    owner_taxpayer_address_entities[
        owner_taxpayer_address_entities[
            "taxpayer_address"
        ].isin(
            candidate_taxpayer_addresses
        )
    ]
    .groupby(
        [
            "taxpayer_address",
            "entity_id",
            "canonical_name",
            "entity_type"
        ],
        dropna=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna()
                            .astype(str)
                        )
                    )
                )
        )
    )
    .reset_index()
    .merge(
        taxpayer_address_profile[
            [
                "taxpayer_address",
                "distinct_entities",
                "distinct_parcels",
                "distinct_person_entities",
                "distinct_organization_entities"
            ]
        ],
        on="taxpayer_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "taxpayer_address",
            "entity_type",
            "canonical_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

taxpayer_address_member_summary

In [ ]:
taxpayer_address_member_summary.to_csv(
    "taxpayer_address_member_summary.csv",
    index=False
)

shared_taxpayer_address_candidates.to_csv(
    "shared_taxpayer_address_candidates.csv",
    index=False
)

In [ ]:
print(
    "Parcels with more than one taxpayer address:",
    (taxpayer_address_counts > 1).sum()
)

print(
    "Owner rows with missing entity type:",
    owner_taxpayer_address_entities[
        "entity_type"
    ].isna().sum()
)

In [ ]:
shared_taxpayer_address_candidates

## Stage 6 pre-implementation audit

In [ ]:
strength_rank = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}


# ----------------------------------------------------------
# Summarize all evidence at the entity-pair level
# ----------------------------------------------------------

pair_evidence_summary = (
    association_evidence
    .groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        dropna=False
    )
    .agg(
        evidence_count=(
            "evidence_id",
            "count"
        ),

        distinct_evidence_rules=(
            "evidence_rule",
            "nunique"
        ),

        evidence_rules=(
            "evidence_rule",
            lambda x:
                tuple(
                    sorted(
                        set(
                            x.dropna()
                        )
                    )
                )
        ),

        evidence_strengths=(
            "evidence_strength",
            lambda x:
                tuple(
                    sorted(
                        set(
                            x.dropna()
                        ),
                        key=lambda value:
                            strength_rank.get(
                                value,
                                0
                            ),
                        reverse=True
                    )
                )
        ),

        strongest_evidence_strength=(
            "evidence_strength",
            lambda x:
                max(
                    x.dropna(),
                    key=lambda value:
                        strength_rank.get(
                            value,
                            0
                        )
                )
        )
    )
    .reset_index()
)


# ----------------------------------------------------------
# Add readable entity names/types
# ----------------------------------------------------------

entity_a_lookup = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "entity_id_a",
            "canonical_name":
                "entity_name_a",
            "entity_type":
                "entity_type_a"
        }
    )
)

entity_b_lookup = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "entity_id_b",
            "canonical_name":
                "entity_name_b",
            "entity_type":
                "entity_type_b"
        }
    )
)

pair_evidence_summary = (
    pair_evidence_summary
    .merge(
        entity_a_lookup,
        on="entity_id_a",
        how="left",
        validate="many_to_one"
    )
    .merge(
        entity_b_lookup,
        on="entity_id_b",
        how="left",
        validate="many_to_one"
    )
)


# ----------------------------------------------------------
# Human-readable rule-combination label
# ----------------------------------------------------------

pair_evidence_summary[
    "rule_combination"
] = (
    pair_evidence_summary[
        "evidence_rules"
    ]
    .apply(
        lambda rules:
            " + ".join(rules)
    )
)

In [ ]:
# ==========================================================
# OVERALL MULTIPLICITY
# ==========================================================

total_pairs = len(
    pair_evidence_summary
)

multiple_evidence_pairs = (
    pair_evidence_summary[
        pair_evidence_summary[
            "evidence_count"
        ] > 1
    ]
    .copy()
)

multiple_rule_pairs = (
    pair_evidence_summary[
        pair_evidence_summary[
            "distinct_evidence_rules"
        ] > 1
    ]
    .copy()
)

repeated_same_rule_pairs = (
    pair_evidence_summary[
        (
            pair_evidence_summary[
                "evidence_count"
            ] > 1
        )
        &
        (
            pair_evidence_summary[
                "distinct_evidence_rules"
            ] == 1
        )
    ]
    .copy()
)

print(
    "Total distinct entity pairs:",
    total_pairs
)

print(
    "Pairs with multiple evidence rows:",
    len(
        multiple_evidence_pairs
    )
)

print(
    "Pairs supported by multiple distinct rules:",
    len(
        multiple_rule_pairs
    )
)

print(
    "Pairs with repeated evidence from only one rule:",
    len(
        repeated_same_rule_pairs
    )
)

print(
    "Maximum evidence rows for one pair:",
    pair_evidence_summary[
        "evidence_count"
    ].max()
)

print(
    "Maximum distinct rules for one pair:",
    pair_evidence_summary[
        "distinct_evidence_rules"
    ].max()
)

In [ ]:
# ==========================================================
# RULE-COMBINATION FREQUENCIES
# ==========================================================

rule_combination_counts = (
    pair_evidence_summary
    .groupby(
        [
            "rule_combination",
            "distinct_evidence_rules"
        ],
        dropna=False
    )
    .agg(
        pair_count=(
            "entity_id_a",
            "count"
        ),

        total_evidence_rows=(
            "evidence_count",
            "sum"
        ),

        min_evidence_rows=(
            "evidence_count",
            "min"
        ),

        max_evidence_rows=(
            "evidence_count",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        [
            "distinct_evidence_rules",
            "pair_count"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

rule_combination_counts

In [ ]:
multi_rule_combination_counts = (
    rule_combination_counts[
        rule_combination_counts[
            "distinct_evidence_rules"
        ] > 1
    ]
    .reset_index(drop=True)
)

print(
    "Distinct multi-rule combinations:",
    len(
        multi_rule_combination_counts
    )
)

multi_rule_combination_counts

In [ ]:
# ==========================================================
# STRENGTH-COMBINATION AUDIT
# ==========================================================

strength_combination_counts = (
    pair_evidence_summary
    .assign(
        strength_combination=
            pair_evidence_summary[
                "evidence_strengths"
            ]
            .apply(
                lambda strengths:
                    " + ".join(
                        strengths
                    )
            )
    )
    .groupby(
        [
            "strongest_evidence_strength",
            "strength_combination"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="pair_count"
    )
    .sort_values(
        [
            "strongest_evidence_strength",
            "pair_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

strength_combination_counts

In [ ]:
# ==========================================================
# MULTI-RULE PAIR SUMMARY
# ==========================================================

multi_rule_pair_summary = (
    multiple_rule_pairs[
        [
            "entity_id_a",
            "entity_name_a",
            "entity_type_a",
            "entity_id_b",
            "entity_name_b",
            "entity_type_b",
            "evidence_count",
            "distinct_evidence_rules",
            "evidence_rules",
            "evidence_strengths",
            "strongest_evidence_strength",
            "rule_combination"
        ]
    ]
    .sort_values(
        [
            "distinct_evidence_rules",
            "evidence_count",
            "entity_name_a",
            "entity_name_b"
        ],
        ascending=[
            False,
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

multi_rule_pair_summary

In [ ]:
# ==========================================================
# DETAILED EVIDENCE FOR MULTI-RULE PAIRS
# ==========================================================

multi_rule_pair_keys = (
    multiple_rule_pairs[
        [
            "entity_id_a",
            "entity_id_b"
        ]
    ]
)

multi_rule_evidence_detail = (
    association_evidence
    .merge(
        multi_rule_pair_keys,
        on=[
            "entity_id_a",
            "entity_id_b"
        ],
        how="inner",
        validate="many_to_one"
    )
    .merge(
        entity_a_lookup,
        on="entity_id_a",
        how="left",
        validate="many_to_one"
    )
    .merge(
        entity_b_lookup,
        on="entity_id_b",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        [
            "entity_id_a",
            "entity_id_b",
            "evidence_strength",
            "evidence_rule",
            "evidence_id"
        ]
    )
    .reset_index(drop=True)
)

multi_rule_evidence_detail

In [ ]:
# ==========================================================
# EVIDENCE-COUNT DISTRIBUTION
# ==========================================================

evidence_count_distribution = (
    pair_evidence_summary[
        "evidence_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "evidence_rows_per_pair"
    )
    .reset_index(
        name="pair_count"
    )
)

evidence_count_distribution

In [ ]:
sum(entity_associations["evidence_count"]) == 491

In [ ]:
# ==========================================================
# POST-CLEANUP STAGE 6 AUDIT SANITY CHECKS
# ==========================================================

assert len(
    pair_evidence_summary
) == 315

assert (
    pair_evidence_summary[
        "evidence_count"
    ].sum()
    == 491
)

assert (
    pair_evidence_summary[
        "strongest_evidence_strength"
    ]
    .isin(
        [
            "High",
            "Medium",
            "Low"
        ]
    )
).all()

assert not pair_evidence_summary.duplicated(
    subset=[
        "entity_id_a",
        "entity_id_b"
    ]
).any()

print(
    "Stage 6 pre-implementation audit "
    "sanity checks passed."
)

## Stage 6 entity association validation

In [ ]:
print(
    "Entity association rows:",
    len(entity_associations)
)

print(
    "\nAssociation confidence:"
)

print(
    entity_associations[
        "association_confidence"
    ].value_counts()
)

print(
    "\nTotal underlying evidence rows:",
    entity_associations[
        "evidence_count"
    ].sum()
)

print(
    "Maximum evidence rows for one association:",
    entity_associations[
        "evidence_count"
    ].max()
)

print(
    "Maximum distinct evidence rules:",
    entity_associations[
        "evidence_rule_count"
    ].max()
)


# ----------------------------------------------------------
# Basic Stage 6 grain / ID validation
# ----------------------------------------------------------

assert len(
    entity_associations
) == 315

assert entity_associations[
    "association_id"
].is_unique

assert not entity_associations.duplicated(
    subset=[
        "entity_id_a",
        "entity_id_b"
    ]
).any()

assert (
    entity_associations[
        "entity_id_a"
    ]
    <
    entity_associations[
        "entity_id_b"
    ]
).all()


# ----------------------------------------------------------
# Every referenced entity must exist
# ----------------------------------------------------------

valid_entity_ids = set(
    entities[
        "entity_id"
    ]
)

assert set(
    entity_associations[
        "entity_id_a"
    ]
).issubset(
    valid_entity_ids
)

assert set(
    entity_associations[
        "entity_id_b"
    ]
).issubset(
    valid_entity_ids
)


# ----------------------------------------------------------
# Evidence counts must reconcile exactly to Stage 5
# ----------------------------------------------------------

assert (
    entity_associations[
        "evidence_count"
    ].sum()
    ==
    len(
        association_evidence
    )
)

assert (
    entity_associations[
        "evidence_count"
    ].sum()
    == 491
)

assert (
    entity_associations[
        "evidence_count"
    ]
    >= 1
).all()

assert (
    entity_associations[
        "evidence_rule_count"
    ]
    >= 1
).all()

assert (
    entity_associations[
        "evidence_rule_count"
    ]
    <=
    entity_associations[
        "evidence_count"
    ]
).all()


# ----------------------------------------------------------
# Expected confidence distribution
# ----------------------------------------------------------

confidence_counts = (
    entity_associations[
        "association_confidence"
    ]
    .value_counts()
    .to_dict()
)

assert confidence_counts == {
    "High": 151,
    "Medium": 127,
    "Low": 37
}


# ----------------------------------------------------------
# Recalculate strongest evidence independently and confirm
# Stage 6 confidence matches it exactly.
# ----------------------------------------------------------

strength_rank = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

rank_strength = {
    1: "Low",
    2: "Medium",
    3: "High"
}

expected_confidence = (
    association_evidence
    .assign(
        _strength_rank=
            association_evidence[
                "evidence_strength"
            ]
            .map(
                strength_rank
            )
    )
    .groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        as_index=False
    )
    .agg(
        expected_strength_rank=(
            "_strength_rank",
            "max"
        )
    )
)

expected_confidence[
    "expected_confidence"
] = (
    expected_confidence[
        "expected_strength_rank"
    ]
    .map(
        rank_strength
    )
)

confidence_check = (
    entity_associations[
        [
            "entity_id_a",
            "entity_id_b",
            "association_confidence"
        ]
    ]
    .merge(
        expected_confidence[
            [
                "entity_id_a",
                "entity_id_b",
                "expected_confidence"
            ]
        ],
        on=[
            "entity_id_a",
            "entity_id_b"
        ],
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

assert (
    confidence_check[
        "_merge"
    ]
    == "both"
).all()

assert (
    confidence_check[
        "association_confidence"
    ]
    ==
    confidence_check[
        "expected_confidence"
    ]
).all()


# ----------------------------------------------------------
# Independently validate evidence_count and rule_count
# ----------------------------------------------------------

expected_counts = (
    association_evidence
    .groupby(
        [
            "entity_id_a",
            "entity_id_b"
        ],
        as_index=False
    )
    .agg(
        expected_evidence_count=(
            "evidence_id",
            "count"
        ),

        expected_rule_count=(
            "evidence_rule",
            "nunique"
        )
    )
)

count_check = (
    entity_associations[
        [
            "entity_id_a",
            "entity_id_b",
            "evidence_count",
            "evidence_rule_count"
        ]
    ]
    .merge(
        expected_counts,
        on=[
            "entity_id_a",
            "entity_id_b"
        ],
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

assert (
    count_check[
        "_merge"
    ]
    == "both"
).all()

assert (
    count_check[
        "evidence_count"
    ]
    ==
    count_check[
        "expected_evidence_count"
    ]
).all()

assert (
    count_check[
        "evidence_rule_count"
    ]
    ==
    count_check[
        "expected_rule_count"
    ]
).all()


print(
    "\nStage 6 entity-association validation passed."
)

## Final database tables audit

In [ ]:
print("CLEAN_DF")
print(clean_df.dtypes)
print(clean_df.columns.tolist())

print("\nENTITIES")
print(entities.dtypes)
print(entities.columns.tolist())

print("\nPARCEL ENTITY ROLES")
print(parcel_entity_roles.dtypes)
print(parcel_entity_roles.columns.tolist())

print("\nASSOCIATION EVIDENCE")
print(association_evidence.dtypes)
print(association_evidence.columns.tolist())

print("\nENTITY ASSOCIATIONS")
print(entity_associations.dtypes)
print(entity_associations.columns.tolist())

print("\nENTITY OCCURRENCES")
print(entity_occurrences.dtypes)
print(entity_occurrences.columns.tolist())

In [ ]:
# ==========================================================
# DATABASE SCHEMA RANGE AUDIT
# ==========================================================

def max_string_length(df, column):
    return (
        df[column]
        .dropna()
        .astype(str)
        .str.len()
        .max()
    )


print("PARCELS")
print(
    "parcel_number max length:",
    max_string_length(
        clean_df,
        "parcel_number"
    )
)

print(
    "property_address max length:",
    max_string_length(
        clean_df,
        "property_address"
    )
)

print(
    "property_class max length:",
    max_string_length(
        clean_df,
        "property_class"
    )
)

print(
    "PRE range:",
    clean_df[
        "principal_residence_exemption_percentage"
    ].min(),
    "to",
    clean_df[
        "principal_residence_exemption_percentage"
    ].max()
)

print(
    "tax_value range:",
    clean_df["tax_value"].min(),
    "to",
    clean_df["tax_value"].max()
)

print(
    "assessed_value range:",
    clean_df["assessed_value"].min(),
    "to",
    clean_df["assessed_value"].max()
)

print(
    "state_equalized_value range:",
    clean_df[
        "state_equalized_value"
    ].min(),
    "to",
    clean_df[
        "state_equalized_value"
    ].max()
)


print("\nENTITIES")

for column in [
    "canonical_name",
    "canonical_address",
    "entity_type",
    "organization_type",
    "organization_classification_confidence"
]:
    print(
        f"{column} max length:",
        max_string_length(
            entities,
            column
        )
    )


print("\nASSOCIATION EVIDENCE")

for column in [
    "evidence_rule",
    "evidence_strength",
    "role_a",
    "role_b",
    "shared_address",
    "address_type"
]:
    print(
        f"{column} max length:",
        max_string_length(
            association_evidence,
            column
        )
    )


print("\nENTITY OCCURRENCES")

for column in [
    "entity_name",
    "associated_address",
    "role",
    "source_entity_name",
    "source_column",
    "owner_line_continuation_rule",
    "entity_type",
    "joint_parse_rule",
    "joint_parse_confidence",
    "organization_type",
    "organization_classification_confidence",
    "address_source"
]:
    print(
        f"{column} max length:",
        max_string_length(
            entity_occurrences,
            column
        )
    )

print(
    "joint_party_index max:",
    entity_occurrences[
        "joint_party_index"
    ].max()
)

print(
    "joint_party_count max:",
    entity_occurrences[
        "joint_party_count"
    ].max()
)

## Owner address audit

In [ ]:
owner_address_entities = (
    entity_occurrences[
        (
            entity_occurrences["role"]
            == "Owner"
        )
        &
        (
            entity_occurrences[
                "associated_address"
            ].notna()
        )
        &
        (
            entity_occurrences[
                "associated_address"
            ].str.strip()
            != ""
        )
    ]
    [
        [
            "parcel_number",
            "entity_id",
            "associated_address"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "associated_address":
                "owner_address"
        }
    )
)

In [ ]:
owner_address_entities = (
    owner_address_entities
    .merge(
        entities[
            [
                "entity_id",
                "entity_type",
                "canonical_name"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
owner_address_profile = (
    owner_address_entities
    .groupby(
        "owner_address"
    )
    .agg(
        distinct_entities=(
            "entity_id",
            "nunique"
        ),
        distinct_parcels=(
            "parcel_number",
            "nunique"
        ),
        distinct_person_entities=(
            "entity_id",
            lambda ids:
                owner_address_entities.loc[
                    ids.index
                ]
                .loc[
                    lambda df:
                        df["entity_type"]
                        == "Person",
                    "entity_id"
                ]
                .nunique()
        ),
        distinct_organization_entities=(
            "entity_id",
            lambda ids:
                owner_address_entities.loc[
                    ids.index
                ]
                .loc[
                    lambda df:
                        df["entity_type"]
                        == "Organization",
                    "entity_id"
                ]
                .nunique()
        )
    )
    .reset_index()
)

owner_address_profile.head(30)

In [ ]:
owner_address_profile[
    [
        "distinct_entities",
        "distinct_parcels"
    ]
].describe()

In [ ]:
owner_address_persons = (
    owner_address_entities[
        owner_address_entities[
            "entity_type"
        ] == "Person"
    ]
    .rename(
        columns={
            "entity_id":
                "person_id",
            "canonical_name":
                "person_name",
            "parcel_number":
                "person_parcel"
        }
    )
)

owner_address_organizations = (
    owner_address_entities[
        owner_address_entities[
            "entity_type"
        ] == "Organization"
    ]
    .rename(
        columns={
            "entity_id":
                "organization_id",
            "canonical_name":
                "organization_name",
            "parcel_number":
                "organization_parcel"
        }
    )
)

shared_owner_address_candidates = (
    owner_address_organizations
    .merge(
        owner_address_persons,
        on="owner_address",
        how="inner"
    )
)

In [ ]:
shared_owner_address_candidates = (
    shared_owner_address_candidates[
        shared_owner_address_candidates[
            "organization_parcel"
        ]
        !=
        shared_owner_address_candidates[
            "person_parcel"
        ]
    ]
    .copy()
)

In [ ]:
print(
    "Cross-parcel shared-owner-address evidence rows:",
    len(shared_owner_address_candidates)
)

print(
    "Distinct Person-Organization pairs:",
    shared_owner_address_candidates[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct shared owner addresses:",
    shared_owner_address_candidates[
        "owner_address"
    ].nunique()
)

In [ ]:
candidate_address_degree = (
    shared_owner_address_candidates[
        [
            "owner_address"
        ]
    ]
    .drop_duplicates()
    .merge(
        owner_address_profile,
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels"
        ],
        ascending=False
    )
)

candidate_address_degree.head(30)

In [ ]:
owner_address_candidate_detail = (
    shared_owner_address_candidates[
        [
            "owner_address",
            "person_id",
            "person_name",
            "person_parcel",
            "organization_id",
            "organization_name",
            "organization_parcel"
        ]
    ]
    .merge(
        owner_address_profile[
            [
                "owner_address",
                "distinct_entities",
                "distinct_parcels"
            ]
        ],
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "owner_address",
            "organization_name",
            "person_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

owner_address_candidate_detail

In [ ]:
person_address_members = (
    owner_address_persons[
        [
            "owner_address",
            "person_id",
            "person_name",
            "person_parcel"
        ]
    ]
    .rename(
        columns={
            "person_id": "entity_id",
            "person_name": "entity_name",
            "person_parcel": "parcel_number"
        }
    )
    .assign(
        entity_type="Person"
    )
)

organization_address_members = (
    owner_address_organizations[
        [
            "owner_address",
            "organization_id",
            "organization_name",
            "organization_parcel"
        ]
    ]
    .rename(
        columns={
            "organization_id": "entity_id",
            "organization_name": "entity_name",
            "organization_parcel": "parcel_number"
        }
    )
    .assign(
        entity_type="Organization"
    )
)

candidate_addresses = set(
    shared_owner_address_candidates[
        "owner_address"
    ]
)

owner_address_members = (
    pd.concat(
        [
            person_address_members,
            organization_address_members
        ],
        ignore_index=True
    )
)

owner_address_members = (
    owner_address_members[
        owner_address_members[
            "owner_address"
        ].isin(candidate_addresses)
    ]
    .drop_duplicates()
)

In [ ]:
owner_address_member_summary = (
    owner_address_members
    .groupby(
        [
            "owner_address",
            "entity_id",
            "entity_name",
            "entity_type"
        ],
        dropna=False
    )
    .agg(
        parcel_count=(
            "parcel_number",
            "nunique"
        ),
        parcels=(
            "parcel_number",
            lambda x:
                ", ".join(
                    sorted(set(x))
                )
        )
    )
    .reset_index()
    .merge(
        owner_address_profile[
            [
                "owner_address",
                "distinct_entities",
                "distinct_parcels"
            ]
        ],
        on="owner_address",
        how="left"
    )
    .sort_values(
        [
            "distinct_entities",
            "distinct_parcels",
            "owner_address",
            "entity_type",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

owner_address_member_summary

## Audit of role combinations between different entities on the same parcel

In [ ]:
parcel_role_pairs = (
    parcel_entity_roles
    .merge(
        parcel_entity_roles,
        on="parcel_number",
        suffixes=("_a", "_b")
    )
)

# Remove self-pairs and keep each unordered entity pair once
parcel_role_pairs = (
    parcel_role_pairs[
        parcel_role_pairs["entity_id_a"]
        < parcel_role_pairs["entity_id_b"]
    ]
    .copy()
)

In [ ]:
entity_lookup_a = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id": "entity_id_a",
            "canonical_name": "name_a",
            "entity_type": "entity_type_a",
            "organization_type": "organization_type_a"
        }
    )
)

entity_lookup_b = (
    entities[
        [
            "entity_id",
            "canonical_name",
            "entity_type",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id": "entity_id_b",
            "canonical_name": "name_b",
            "entity_type": "entity_type_b",
            "organization_type": "organization_type_b"
        }
    )
)

parcel_role_pairs = (
    parcel_role_pairs
    .merge(
        entity_lookup_a,
        on="entity_id_a",
        how="left"
    )
    .merge(
        entity_lookup_b,
        on="entity_id_b",
        how="left"
    )
)

In [ ]:
parcel_role_pairs["role_pair"] = (
    parcel_role_pairs[
        ["role_a", "role_b"]
    ]
    .apply(
        lambda row: " + ".join(
            sorted(row)
        ),
        axis=1
    )
)

In [ ]:
parcel_role_pairs["entity_type_pair"] = (
    parcel_role_pairs[
        [
            "entity_type_a",
            "entity_type_b"
        ]
    ]
    .apply(
        lambda row: " + ".join(
            sorted(row)
        ),
        axis=1
    )
)

In [ ]:
role_pair_counts = (
    parcel_role_pairs[
        "role_pair"
    ]
    .value_counts()
    .rename_axis("role_pair")
    .reset_index(name="count")
)

role_pair_counts

In [ ]:
role_type_pair_counts = (
    parcel_role_pairs
    .groupby(
        [
            "role_pair",
            "entity_type_pair"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

role_type_pair_counts

In [ ]:
print(
    "Parcels containing more than one entity:",
    parcel_role_pairs[
        "parcel_number"
    ].nunique()
)

print(
    "Distinct same-parcel entity pairs:",
    parcel_role_pairs[
        [
            "parcel_number",
            "entity_id_a",
            "entity_id_b"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
org_person_pairs = (
    parcel_role_pairs[
        (
            (
                parcel_role_pairs["entity_type_a"]
                == "Organization"
            )
            &
            (
                parcel_role_pairs["entity_type_b"]
                == "Person"
            )
        )
        |
        (
            (
                parcel_role_pairs["entity_type_a"]
                == "Person"
            )
            &
            (
                parcel_role_pairs["entity_type_b"]
                == "Organization"
            )
        )
    ]
    .copy()
)

In [ ]:
org_person_pairs["organization_id"] = (
    org_person_pairs.apply(
        lambda row:
            row["entity_id_a"]
            if row["entity_type_a"] == "Organization"
            else row["entity_id_b"],
        axis=1
    )
)

org_person_pairs["organization_name"] = (
    org_person_pairs.apply(
        lambda row:
            row["name_a"]
            if row["entity_type_a"] == "Organization"
            else row["name_b"],
        axis=1
    )
)

org_person_pairs["organization_role"] = (
    org_person_pairs.apply(
        lambda row:
            row["role_a"]
            if row["entity_type_a"] == "Organization"
            else row["role_b"],
        axis=1
    )
)

org_person_pairs["person_id"] = (
    org_person_pairs.apply(
        lambda row:
            row["entity_id_a"]
            if row["entity_type_a"] == "Person"
            else row["entity_id_b"],
        axis=1
    )
)

org_person_pairs["person_name"] = (
    org_person_pairs.apply(
        lambda row:
            row["name_a"]
            if row["entity_type_a"] == "Person"
            else row["name_b"],
        axis=1
    )
)

org_person_pairs["person_role"] = (
    org_person_pairs.apply(
        lambda row:
            row["role_a"]
            if row["entity_type_a"] == "Person"
            else row["role_b"],
        axis=1
    )
)

In [ ]:
org_person_role_counts = (
    org_person_pairs
    .groupby(
        [
            "organization_role",
            "person_role"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
    .reset_index(drop=True)
)

org_person_role_counts

In [ ]:
print(
    "Organization-Person evidence rows:",
    len(org_person_pairs)
)

print(
    "Distinct parcel/person/organization combinations:",
    org_person_pairs[
        [
            "parcel_number",
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct person/organization pairs across all parcels:",
    org_person_pairs[
        [
            "person_id",
            "organization_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
org_person_detail = (
    org_person_pairs[
        [
            "parcel_number",

            "person_id",
            "person_name",
            "person_role",

            "organization_id",
            "organization_name",
            "organization_type_a",
            "organization_type_b",
            "organization_role"
        ]
    ]
    .copy()
)

organization_lookup = (
    entities[
        [
            "entity_id",
            "organization_type"
        ]
    ]
    .rename(
        columns={
            "entity_id":
                "organization_id"
        }
    )
)

org_person_detail = (
    org_person_pairs[
        [
            "parcel_number",

            "person_id",
            "person_name",
            "person_role",

            "organization_id",
            "organization_name",
            "organization_role"
        ]
    ]
    .merge(
        organization_lookup,
        on="organization_id",
        how="left"
    )
    .sort_values(
        [
            "organization_role",
            "person_role",
            "parcel_number"
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# Get the six most common directional role combinations
top_role_combinations = (
    org_person_role_counts
    .head(6)[
        [
            "organization_role",
            "person_role"
        ]
    ]
)

# Keep only rows belonging to those six combinations
top_role_candidates = (
    org_person_detail
    .merge(
        top_role_combinations,
        on=[
            "organization_role",
            "person_role"
        ],
        how="inner"
    )
)

# Randomize the order reproducibly, then take up to
# 10 examples from each role combination.
top_role_samples = (
    top_role_candidates
    .sample(
        frac=1,
        random_state=42
    )
    .groupby(
        [
            "organization_role",
            "person_role"
        ],
        group_keys=False,
        sort=False
    )
    .head(10)
    .sort_values(
        [
            "organization_role",
            "person_role",
            "parcel_number"
        ]
    )
    .reset_index(drop=True)
)

top_role_samples[
    [
        "parcel_number",
        "person_name",
        "person_role",
        "organization_name",
        "organization_type",
        "organization_role"
    ]
]

## Investigating person-like entities

In [ ]:
person_red_flag_pattern = (
    r"\b(?:"
    r"IRA|"
    r"FBO|"
    r"CUSTODIAN|"
    r"SOLE\s+PROPRIETORSHIP|"
    r"MGMT|"
    r"MANAGEMENT|"
    r"MORTGAGE|"
    r"BANK|"
    r"PROPERTIES|"
    r"HOLDINGS|"
    r"SERV|"
    r"SERVICES"
    r")\b"
)

person_classification_audit = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.contains(
                person_red_flag_pattern,
                regex=True,
                na=False
            )
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Potentially suspicious Person entities:",
    len(person_classification_audit)
)

person_classification_audit

In [ ]:
suspicious_names = [
    "FBO GARY E APPS IRA",
    "FBO JOHN MEYERS IRA",
    "FBO RAMON HUERTA IRA",
    "FBO RICHARD DAUDERT IRA",
    "RUSHMORE LOAD MGT SERV"
]

entities[
    entities["canonical_name"].isin(
        suspicious_names
    )
][
    [
        "entity_id",
        "canonical_name",
        "entity_type",
        "organization_type",
        "organization_classification_confidence",
        "canonical_address",
        "occurrence_count"
    ]
].sort_values(
    [
        "canonical_name",
        "canonical_address"
    ]
)

In [ ]:
entities["occurrence_count"].sum()

## Auditing multi-comma persons

In [ ]:
multi_comma_persons = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.count(",")
            >= 3
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Person entities with at least 3 commas:",
    len(multi_comma_persons)
)

multi_comma_persons

In [ ]:
two_comma_persons = (
    entities[
        (
            entities["entity_type"]
            == "Person"
        )
        &
        (
            entities["canonical_name"]
            .str.count(",")
            == 2
        )
    ]
    [
        [
            "entity_id",
            "canonical_name",
            "canonical_address",
            "occurrence_count"
        ]
    ]
    .sort_values(
        "canonical_name"
    )
    .reset_index(drop=True)
)

print(
    "Person entities with exactly 2 commas:",
    len(two_comma_persons)
)

two_comma_persons

## Audit parcel_entity_roles

In [ ]:
print(parcel_entity_roles.head(10))

In [ ]:
print(
    "Entity occurrences:",
    len(entity_occurrences)
)

print(
    "Parcel entity roles:",
    len(parcel_entity_roles)
)

print(
    "Unique parcels:",
    parcel_entity_roles[
        "parcel_number"
    ].nunique()
)

print(
    "Unique entities represented:",
    parcel_entity_roles[
        "entity_id"
    ].nunique()
)

In [ ]:
duplicate_relationships = (
    parcel_entity_roles
    .duplicated(
        subset=[
            "parcel_number",
            "entity_id",
            "role"
        ]
    )
    .sum()
)

print(
    "Duplicate parcel/entity/role relationships:",
    duplicate_relationships
)

In [ ]:
expected_relationships = (
    entity_occurrences[
        [
            "parcel_number",
            "entity_id",
            "role"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Expected relationships:",
    len(expected_relationships)
)

print(
    "Actual relationships:",
    len(parcel_entity_roles)
)

assert (
    len(expected_relationships)
    == len(parcel_entity_roles)
)

In [ ]:
parcel_entity_roles[
    "role"
].value_counts()

In [ ]:
parcel_entity_roles.groupby(
    "parcel_number"
)["entity_id"].nunique().describe()

In [ ]:
role_occurrence_counts = (
    entity_occurrences
    .groupby(
        [
            "parcel_number",
            "entity_id",
            "role"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="occurrence_count"
    )
)

duplicate_role_relationships = (
    role_occurrence_counts[
        role_occurrence_counts[
            "occurrence_count"
        ] > 1
    ]
    .sort_values(
        by="occurrence_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Parcel/entity/role relationships represented "
    "by multiple occurrences:",
    len(duplicate_role_relationships)
)

duplicate_role_relationships

In [ ]:
duplicate_role_detail = (
    entity_occurrences
    .merge(
        duplicate_role_relationships[
            [
                "parcel_number",
                "entity_id",
                "role"
            ]
        ],
        on=[
            "parcel_number",
            "entity_id",
            "role"
        ],
        how="inner"
    )
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address"
            ]
        ],
        on="entity_id",
        how="left"
    )
    .sort_values(
        by=[
            "parcel_number",
            "entity_id",
            "role",
            "occurrence_id"
        ]
    )
)

duplicate_role_detail[
    [
        "parcel_number",
        "entity_id",
        "canonical_name",
        "entity_type",
        "role",
        "entity_name",
        "source_entity_name",
        "source_column",
        "associated_address",
        "joint_expanded",
        "source_occurrence_id",
        "occurrence_id"
    ]
]

In [ ]:
print(
    "Final entities:",
    len(entities)
)

print(
    "Entities represented in parcel roles:",
    parcel_entity_roles[
        "entity_id"
    ].nunique()
)

## Auditing merge_on_match_key

In [ ]:
raw_entities_audit = (
    resolver._create_raw_entities(
        entity_occurrences
    )
)

print(
    "Raw entities:",
    len(raw_entities_audit)
)

In [ ]:
pre_abbreviation_entities = (
    raw_entities_audit.copy()
)

pass_number = 0

while True:

    pass_number += 1

    count_before = len(
        pre_abbreviation_entities
    )

    pre_abbreviation_entities = (
        resolver._merge_blank_addresses(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_missing_street_suffix_variants(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_zip4_variants(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_equivalent_person_name_formats(
            pre_abbreviation_entities
        )
    )

    pre_abbreviation_entities = (
        resolver._merge_middle_initial_variants(
            pre_abbreviation_entities
        )
    )

    count_after = len(
        pre_abbreviation_entities
    )

    print(
        f"Pass {pass_number}: "
        f"{count_before} -> {count_after}"
    )

    if count_after == count_before:
        break

In [ ]:
abbreviation_audit = (
    pre_abbreviation_entities.copy()
)

abbreviation_audit[
    "match_name"
] = (
    abbreviation_audit[
        "canonical_name"
    ]
    .apply(
        resolver._expand_common_abbreviations
    )
)

In [ ]:
abbreviation_candidates = []

for _, group in abbreviation_audit.groupby(
    [
        "entity_type",
        "match_name",
        "canonical_address"
    ],
    dropna=False,
    sort=False
):

    if len(group) < 2:
        continue

    original_names = tuple(
        sorted(
            set(
                group["canonical_name"]
            )
        )
    )

    # Ignore exact duplicates. We only want groups where
    # abbreviation expansion reconciles different names.
    if len(original_names) < 2:
        continue

    abbreviation_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "match_name":
            group["match_name"].iloc[0],

        "canonical_address":
            group["canonical_address"].iloc[0],

        "canonical_names":
            original_names,

        "entity_count":
            len(group),

        "occurrence_count":
            group[
                "occurrence_count"
            ].sum(),

        "row_indexes":
            tuple(
                group.index
            )
    })


abbreviation_candidates = pd.DataFrame(
    abbreviation_candidates
)

print(
    "Common-abbreviation candidate groups:",
    len(abbreviation_candidates)
)

abbreviation_candidates

In [ ]:
if not abbreviation_candidates.empty:

    print(
        abbreviation_candidates[
            "entity_type"
        ].value_counts()
    )

## Entities with the highest occurrence_count

In [ ]:
entities.sort_values(
    "occurrence_count",
    ascending=False
).head(10)

## Entities with canonical names containing a middle name or middle initial

In [ ]:
SUFFIXES = {
    "JR",
    "SR",
    "JUNIOR",
    "SENIOR",
    "II",
    "III",
    "IV",
}


def get_middle_tokens(name):
    """
    Return possible middle-name tokens from a person name.

    Supports both:
        SMITH, JOHN A
        SMITH JOHN A

    This is for manual inspection only, not entity-resolution logic.
    """

    if pd.isna(name):
        return tuple()

    name = str(name).upper().strip()
    name = name.replace(".", "")

    # Skip obvious joint/compound records
    if (
        "&" in name
        or "/" in name
        or re.search(r"\bAND\b", name)
    ):
        return tuple()

    if "," in name:

        _, given_part = name.split(",", 1)
        tokens = given_part.strip().split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # FIRST [MIDDLE...]
        return tuple(tokens[1:])

    else:

        tokens = name.split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # Assume LAST FIRST [MIDDLE...]
        if len(tokens) < 3:
            return tuple()

        return tuple(tokens[2:])

people_with_middle_names = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

people_with_middle_names["middle_tokens"] = (
    people_with_middle_names[
        "canonical_name"
    ]
    .apply(get_middle_tokens)
)

people_with_middle_names = (
    people_with_middle_names[
        people_with_middle_names[
            "middle_tokens"
        ].apply(len) > 0
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

people_with_middle_names[
    [
        "entity_id",
        "canonical_name",
        "middle_tokens",
        "canonical_address",
        "occurrence_count"
    ]
]

- Some names have commas while others don't
    - For example, KAVANAUGH, JAMES J vs. CLAWSON DORRINDA L
        - Note that both had their middle initial listed correctly under middle_tokens
- The canonical name FRITH, TERRANCE JEROME A produced the middle_token (JEROME, A)
- The canonical name BROWN, ROBERT WENDELL, JR. produced the middle_token (WENDELL,,)
    - Note that there are two commas rather than the usual one
- GREATER TRUE WORSHIP COGIC produced (WORSHIP, COGIC)
- BLACKSTAR STABILITY REVITALIZATION produced (REVITALIZATION,)
- EDWARDS, KARRIE; EDWARDS, KENTON produced (EDWARDS,, KENTON)

Unrelated case:
There are two entities with the name WIDGER JAMES LEROY, one with the address 214 MARSHALL ST, ALLEGAN, MI 49010 and the other with the address 214 MARSHALL ST, ALLEGAN, MI 49010. Because they have the exact same name and their address, with the removal of ST from the first case, are the same - they should clearly be merged.

### Distinguishing middle initials from full middle names

In [ ]:
def middle_type(tokens):

    if not tokens:
        return None

    if all(len(token) == 1 for token in tokens):
        return "Initial"

    return "Full/Other"


people_with_middle_names["middle_type"] = (
    people_with_middle_names[
        "middle_tokens"
    ]
    .apply(middle_type)
)

people_with_middle_names[
    "middle_type"
].value_counts()

Out of 3,955 unique entity names, 1,240 show as containing a middle name or initial.

In [ ]:
797 + 443

## Entities the previously had a blank address

In [ ]:
blank_address_cases = (
    resolution_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ],
        dropna=False
    )
    .agg(
        had_blank_address=(
            "associated_address",
            lambda x: (
                x.fillna("")
                .str.strip()
                .eq("")
                .any()
            )
        ),
        source_addresses=(
            "associated_address",
            lambda x: tuple(
                sorted(
                    {
                        address
                        for address in x.fillna("")
                        if address != ""
                    }
                )
            )
        ),
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        occurrence_count=(
            "occurrence_id",
            "size"
        )
    )
    .reset_index()
)

blank_address_cases = (
    blank_address_cases[
        blank_address_cases["had_blank_address"]
        &
        (
            blank_address_cases[
                "canonical_address"
            ] != ""
        )
    ]
)

print(blank_address_cases.shape)
blank_address_cases.head(50)

There are 1,580 cases of entities that used to have a blank address
- **QUESTION: Is the main thing to look for here that the source_names and canonical_name are in fact similar enough that we would want to merge them?**

## Names affected by CORP/CORPORATION, INC/INCORPORATED, CO/COMPANY

In [ ]:
ENTITY_NAME_REPLACEMENTS = {
    "CORP": "CORPORATION",
    "INC": "INCORPORATED",
    "CO": "COMPANY"
}


def expand_entity_abbreviations(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    name = name.replace(".", "")

    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\s*,\s*", ", ", name)

    for old, new in ENTITY_NAME_REPLACEMENTS.items():

        name = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            name
        )

    return name.strip()

abbreviation_audit = resolution_audit.copy()

abbreviation_audit["expanded_name"] = (
    abbreviation_audit[
        "entity_name"
    ]
    .apply(expand_entity_abbreviations)
)

abbreviation_cases = (
    abbreviation_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        expanded_names=(
            "expanded_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        source_name_count=(
            "entity_name",
            "nunique"
        ),
        expanded_name_count=(
            "expanded_name",
            "nunique"
        )
    )
    .reset_index()
)

abbreviation_cases = (
    abbreviation_cases[
        (abbreviation_cases["source_name_count"] > 1)
        &
        (abbreviation_cases["expanded_name_count"] == 1)
    ]
)

abbreviation_cases.head(50)

- There is 1 case of a CORP/CORPORATION merger.
- Other than that, the other 5 cases involve merging source names that differ by the inclusion/exclusion of a period.
    - For example, ONE DAY, LLC and ONE DAY, LLC.

## Suffix cases

### All people containing a suffix

In [ ]:
suffix_pattern = (
    r"\b(?:JR|SR|JUNIOR|SENIOR|II|III|IV)\b"
)

suffix_entities = (
    entities[
        (entities["entity_type"] == "Person")
        &
        (
            entities["canonical_name"]
            .str.contains(
                suffix_pattern,
                regex=True,
                na=False
            )
        )
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

print(suffix_entities.shape)

suffix_entities[
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
].head(50)

- 146 entities contain suffixes

### Entities with both suffix and suffix-free versions

In [ ]:
suffix_audit = (
    resolution_audit[
        resolution_audit[
            "entity_type_final"
        ] == "Person"
    ]
    .copy()
)

suffix_audit["has_suffix"] = (
    suffix_audit[
        "entity_name"
    ]
    .str.contains(
        suffix_pattern,
        regex=True,
        na=False
    )
)

suffix_merge_cases = (
    suffix_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        has_suffix_version=(
            "has_suffix",
            "any"
        ),
        has_no_suffix_version=(
            "has_suffix",
            lambda x: (~x).any()
        )
    )
    .reset_index()
)

suffix_merge_cases = (
    suffix_merge_cases[
        suffix_merge_cases["has_suffix_version"]
        &
        suffix_merge_cases["has_no_suffix_version"]
    ]
)

suffix_merge_cases.head(50)

- There is 1 case of an entity merged due to having the same name and address, with different suffixes
    - Albert Derring and Albert Derring III

## Cases of finalized entities with the same name but different associated addresses

In [ ]:
person_entities = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

known_address_people = (
    person_entities[
        person_entities[
            "canonical_address"
        ].fillna("").str.strip() != ""
    ]
)

common_names_multiple_addresses = (
    known_address_people
    .groupby("canonical_name")
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),
        address_count=(
            "canonical_address",
            "nunique"
        ),
        addresses=(
            "canonical_address",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        entity_ids=(
            "entity_id",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        total_occurrences=(
            "occurrence_count",
            "sum"
        )
    )
    .reset_index()
)

common_names_multiple_addresses = (
    common_names_multiple_addresses[
        common_names_multiple_addresses[
            "address_count"
        ] > 1
    ]
    .sort_values(
        [
            "address_count",
            "total_occurrences"
        ],
        ascending=False
    )
)

print(common_names_multiple_addresses.shape)
common_names_multiple_addresses.head(50)

- There are 296 canonical_names which are associated with more than one entity/address

## Inspecting name formatting

In [ ]:
people = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Person"
    ]
)

comma_names = (
    people["entity_name"]
    .str.contains(",", regex=False, na=False)
)

print(
    f"Person occurrences with comma-format names: "
    f"{comma_names.sum():,}"
)

print(
    f"Person occurrences without comma-format names: "
    f"{(~comma_names).sum():,}"
)

print(
    f"Percent parseable by current comma requirement: "
    f"{comma_names.mean():.1%}"
)

## Inspecting unknown entity types

In [ ]:
entity_occurrences[
    "entity_type"
].value_counts()

In [ ]:
unknown_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Unknown",
        "entity_name"
    ]
    .value_counts()
    .rename_axis("entity_name")
    .reset_index(name="occurrence_count")
)

print(
    f"Unique Unknown names: "
    f"{len(unknown_names):,}"
)

unknown_names.head(100)

In [ ]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences[
            "entity_type"
        ] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
)

joint_names.sample(
    min(100, len(joint_names)),
    random_state=42
).sort_values()

In [ ]:
joint_summary = pd.Series({
    "Contains &": joint_names.str.contains(
        "&",
        regex=False
    ).sum(),

    "Contains AND": joint_names.str.contains(
        r"\bAND\b",
        regex=True
    ).sum(),

    "Contains /": joint_names.str.contains(
        "/",
        regex=False
    ).sum(),

    "Contains comma": joint_names.str.contains(
        ",",
        regex=False
    ).sum(),
})

joint_summary

## Inspecting Joint type entities

In [ ]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

def profile_joint_name(name):

    candidate = resolver._normalize_entity_type_candidate(name)

    return pd.Series({
        "ampersand_count": candidate.count("&"),
        "semicolon_count": candidate.count(";"),
        "slash_count": candidate.count("/"),

        "and_count": len(
            re.findall(
                r"\bAND\b",
                candidate
            )
        ),

        "comma_count": candidate.count(","),

        "contains_et_al": bool(
            re.search(
                r"\bET AL\b",
                candidate
            )
        ),

        "token_count": len(
            candidate.split()
        )
    })

joint_profile = (
    joint_names
    .to_frame(name="entity_name")
)

joint_profile = pd.concat(
    [
        joint_profile,

        joint_profile["entity_name"]
        .apply(profile_joint_name)
    ],
    axis=1
)

joint_profile.head()

In [ ]:
joint_profile["separator_type"] = "Other"

joint_profile.loc[
    (
        joint_profile["ampersand_count"] == 1
    )
    &
    (
        joint_profile[
            [
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) == 0
    ),
    "separator_type"
] = "Single ampersand"

joint_profile.loc[
    joint_profile["semicolon_count"] > 0,
    "separator_type"
] = "Semicolon"

joint_profile.loc[
    joint_profile["slash_count"] > 0,
    "separator_type"
] = "Slash"

joint_profile.loc[
    joint_profile["and_count"] > 0,
    "separator_type"
] = "AND"

joint_profile.loc[
    (
        joint_profile[
            [
                "ampersand_count",
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) > 1
    ),
    "separator_type"
] = "Multiple separators"

joint_profile["separator_type"].value_counts()

In [ ]:
simple_ampersand = (
    joint_profile[
        (
            joint_profile["separator_type"]
            == "Single ampersand"
        )
        &
        (~joint_profile["contains_et_al"])
    ]
)

print(
    f"Unique simple ampersand Joint names: "
    f"{len(simple_ampersand):,}"
)

simple_ampersand[
    [
        "entity_name",
        "comma_count",
        "token_count"
    ]
].sample(
    min(100, len(simple_ampersand)),
    random_state=42
)

In [ ]:
simple_ampersand["comma_count"].value_counts().sort_index()

In [ ]:
simple_ampersand.groupby(
    "comma_count"
)["token_count"].describe()

In [ ]:
print(len(simple_ampersand))

print(
    simple_ampersand["comma_count"]
    .isna()
    .sum()
)

In [ ]:
medium_joint_candidates = (
    entity_occurrences[
        (entity_occurrences["entity_type"] == "Joint")
        &
        (
            entity_occurrences["joint_parse_confidence"]
            == "Medium"
        )
    ]
    .copy()
)

print(
    "Medium-confidence Joint occurrences:",
    f"{len(medium_joint_candidates):,}"
)

print(
    "Unique Medium-confidence Joint names:",
    f"{medium_joint_candidates['entity_name'].nunique():,}"
)

In [ ]:
medium_joint_candidates[
    "joint_parse_rule"
].value_counts()

In [ ]:
medium_joint_names = (
    medium_joint_candidates[
        [
            "entity_name",
            "joint_parse_rule"
        ]
    ]
    .drop_duplicates()
)

medium_joint_names.sample(
    min(100, len(medium_joint_names)),
    random_state=42
).sort_values(
    "entity_name"
)

In [ ]:
expanded_joint_audit = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    [
        [
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "joint_party_index",
            "joint_party_count",
            "joint_parse_rule",
            "joint_parse_confidence"
        ]
    ]
    .sort_values(
        [
            "source_occurrence_id",
            "joint_party_index"
        ]
    )
)

expanded_joint_audit.head(100)

In [ ]:
expanded_joint_pairs = (
    expanded_joint_audit
    .pivot_table(
        index=[
            "source_occurrence_id",
            "source_entity_name",
            "joint_parse_rule",
            "joint_parse_confidence"
        ],
        columns="joint_party_index",
        values="entity_name",
        aggfunc="first"
    )
    .reset_index()
    .rename(
        columns={
            1: "person_1",
            2: "person_2"
        }
    )
)

medium_joint_names.sort_values(
    "entity_name"
).reset_index(
    drop=True
)

In [ ]:
expanded_counts = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    .groupby(
        "source_occurrence_id"
    )
    .size()
)

print(
    "Expanded source occurrences not producing exactly 2 rows:",
    (expanded_counts != 2).sum()
)

In [ ]:
expanded_source_ids = set(
    entity_occurrences.loc[
        entity_occurrences["joint_expanded"],
        "source_occurrence_id"
    ]
)

surviving_joint_rows = (
    entity_occurrences[
        entity_occurrences["source_occurrence_id"]
        .isin(expanded_source_ids)
        &
        (
            entity_occurrences["entity_type"]
            == "Joint"
        )
    ]
)

print(
    "Expanded sources still containing a Joint row:",
    len(surviving_joint_rows)
)

In [ ]:
entity_occurrences["entity_type"].value_counts()

## Analyzing Joint type entities after splitting high and medium confidence one and two comma Joint entities

In [ ]:
remaining_joint = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Joint"
    ]
    .copy()
)

print(
    "Remaining Joint occurrences:",
    f"{len(remaining_joint):,}"
)

print(
    "Unique remaining Joint names:",
    f"{remaining_joint['entity_name'].nunique():,}"
)

In [ ]:
remaining_joint_names = (
    remaining_joint["entity_name"]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
remaining_parseable = (
    remaining_joint_names
    .apply(
        lambda name:
        resolver._parse_joint_name(name)
        is not None
    )
)

print(
    "Remaining Joint names still parseable "
    "by current parser:",
    remaining_parseable.sum()
)

In [ ]:
def profile_joint_name(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    return pd.Series({
        "ampersand_count":
            candidate.count("&"),

        "semicolon_count":
            candidate.count(";"),

        "slash_count":
            candidate.count("/"),

        "and_count":
            len(
                re.findall(
                    r"\bAND\b",
                    candidate
                )
            ),

        "comma_count":
            candidate.count(","),

        "contains_et_al":
            bool(
                re.search(
                    r"\bET AL\b",
                    candidate
                )
            ),

        "token_count":
            len(candidate.split())
    })

In [ ]:
remaining_joint_profile = (
    remaining_joint_names
    .to_frame(name="entity_name")
)

remaining_joint_profile = pd.concat(
    [
        remaining_joint_profile,

        remaining_joint_profile[
            "entity_name"
        ].apply(
            profile_joint_name
        )
    ],
    axis=1
)

In [ ]:
remaining_joint_profile[
    "comma_count"
].value_counts().sort_index()

In [ ]:
remaining_joint_profile[
    [
        "ampersand_count",
        "semicolon_count",
        "slash_count",
        "and_count"
    ]
].describe()

In [ ]:
zero_comma_single_ampersand = (
    remaining_joint_profile[
        (remaining_joint_profile["ampersand_count"] == 1)
        &
        (remaining_joint_profile["comma_count"] == 0)
        &
        (remaining_joint_profile["semicolon_count"] == 0)
        &
        (remaining_joint_profile["slash_count"] == 0)
        &
        (remaining_joint_profile["and_count"] == 0)
        &
        (~remaining_joint_profile["contains_et_al"])
    ]
    .copy()
)

print(
    "Unique zero-comma single-ampersand names:",
    f"{len(zero_comma_single_ampersand):,}"
)

In [ ]:
zero_comma_single_ampersand[
    [
        "entity_name",
        "token_count"
    ]
].sample(
    min(
        100,
        len(zero_comma_single_ampersand)
    ),
    random_state=42
).sort_values(
    "entity_name"
)

## Analyzing single ampersand cases with no commas

In [ ]:
def profile_zero_comma_joint(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    left, right = [
        part.strip()
        for part in candidate.split("&", 1)
    ]

    left_tokens = left.split()
    right_tokens = right.split()

    return pd.Series({
        "left": left,
        "right": right,

        "left_token_count":
            len(left_tokens),

        "right_token_count":
            len(right_tokens),

        "left_empty":
            len(left_tokens) == 0,

        "right_empty":
            len(right_tokens) == 0,

        "left_first_token":
            left_tokens[0]
            if left_tokens
            else "",

        "right_first_token":
            right_tokens[0]
            if right_tokens
            else "",

        "right_last_token":
            right_tokens[-1]
            if right_tokens
            else "",

        "right_ends_initial":
            bool(
                right_tokens
                and len(right_tokens[-1]) == 1
            ),

        "same_leading_token":
            bool(
                left_tokens
                and right_tokens
                and left_tokens[0]
                == right_tokens[0]
            )
    })


zero_comma_structure = pd.concat(
    [
        zero_comma_single_ampersand[
            ["entity_name"]
        ].reset_index(drop=True),

        zero_comma_single_ampersand[
            "entity_name"
        ]
        .reset_index(drop=True)
        .apply(profile_zero_comma_joint)
    ],
    axis=1
)

In [ ]:
zero_comma_structure[
    "right_token_count"
].value_counts().sort_index()

In [ ]:
pd.crosstab(
    zero_comma_structure[
        "right_token_count"
    ],
    zero_comma_structure[
        "right_ends_initial"
    ]
)

In [ ]:
print(
    "Trailing-empty records:",
    zero_comma_structure[
        "right_empty"
    ].sum()
)

print(
    "Same leading token on both sides:",
    zero_comma_structure[
        "same_leading_token"
    ].sum()
)

### Continuing analysis

In [ ]:
pd.crosstab(
    zero_comma_structure["left_token_count"],
    zero_comma_structure["right_token_count"]
)

left_one_token = (
    zero_comma_structure[
        (zero_comma_structure["left_token_count"] == 1)
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

print(
    "Unique names with one token on the left:",
    len(left_one_token)
)

left_one_token[
    [
        "entity_name",
        "left",
        "right",
        "right_token_count"
    ]
].sort_values(
    "entity_name"
)

In [ ]:
right_two_ending_initial = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (zero_comma_structure["right_ends_initial"])
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

right_two_ending_initial[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

In [ ]:
right_two_full_tokens = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (~zero_comma_structure["right_ends_initial"])
    ]
    .copy()
)

right_two_full_tokens[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

In [ ]:
entity_occurrences.loc[
    entity_occurrences["joint_expanded"],
    "joint_parse_rule"
].value_counts()

## Investigating building a name dictionary from county data

In [ ]:
known_first_names = set()
known_last_names = set()

for name in (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Person",
        "entity_name"
    ]
    .drop_duplicates()
):

    normalized = resolver._normalize_match_name(
        name
    )

    # Only use comma-format names for building the vocabulary.
    # Their name orientation is explicitly represented in the
    # source rather than inferred.
    if "," not in normalized:
        continue

    parsed = resolver._parse_person_name(
        normalized
    )

    if parsed is None:
        continue

    known_first_names.add(
        parsed["first_name"]
    )

    known_last_names.add(
        parsed["last_name"]
    )

In [ ]:
def get_name_position_evidence(row):

    right_tokens = row["right"].split()

    if len(right_tokens) != 2:
        return pd.Series({
            "right_token_1": "",
            "right_token_2": "",
            "token_1_seen_first": False,
            "token_1_seen_last": False,
            "token_2_seen_first": False,
            "token_2_seen_last": False
        })

    token_1, token_2 = right_tokens

    return pd.Series({
        "right_token_1": token_1,
        "right_token_2": token_2,

        "token_1_seen_first":
            token_1 in known_first_names,

        "token_1_seen_last":
            token_1 in known_last_names,

        "token_2_seen_first":
            token_2 in known_first_names,

        "token_2_seen_last":
            token_2 in known_last_names
    })


right_two_full_evidence = pd.concat(
    [
        right_two_full_tokens.reset_index(
            drop=True
        ),

        right_two_full_tokens.reset_index(
            drop=True
        ).apply(
            get_name_position_evidence,
            axis=1
        )
    ],
    axis=1
)

In [ ]:
def classify_right_side_evidence(row):

    # Strong evidence for:
    #
    # SURNAME FIRST
    #
    # e.g. MCCADEN SHAWNA
    if (
        row["token_1_seen_last"]
        and row["token_2_seen_first"]
        and not row["token_1_seen_first"]
    ):
        return "Likely separate surname"

    # Strong evidence for:
    #
    # FIRST MIDDLE
    #
    # e.g. MARY ANNE
    if (
        row["token_1_seen_first"]
        and not row["token_1_seen_last"]
        and not row["token_2_seen_last"]
    ):
        return "Likely shared surname"

    return "Ambiguous"


right_two_full_evidence[
    "name_evidence"
] = right_two_full_evidence.apply(
    classify_right_side_evidence,
    axis=1
)

In [ ]:
right_two_full_evidence[
    "name_evidence"
].value_counts()

In [ ]:
right_two_full_evidence[
    [
        "entity_name",
        "right",
        "right_token_1",
        "right_token_2",
        "token_1_seen_first",
        "token_1_seen_last",
        "token_2_seen_first",
        "token_2_seen_last",
        "name_evidence"
    ]
].sort_values(
    [
        "name_evidence",
        "entity_name"
    ]
)

### Adding address data

In [ ]:
person_evidence_rows = []

for _, row in entity_occurrences.iterrows():

    # Avoid using Joint-derived people to prove our own Joint
    # decomposition assumptions.
    if row["entity_type"] != "Person":
        continue

    if row.get("joint_expanded", False):
        continue

    parsed = resolver._parse_person_name(
        row["entity_name"]
    )

    if parsed is None:
        continue

    person_evidence_rows.append({
        "entity_name": row["entity_name"],
        "associated_address": row["associated_address"],
        "last_name": parsed["last_name"],
        "first_name": parsed["first_name"],
        "middle_tokens": parsed["middle_tokens"],
        "suffix": parsed["suffix"]
    })


person_evidence = pd.DataFrame(
    person_evidence_rows
)

In [ ]:
def evaluate_two_full_token_joint(row):

    left_tokens = row["left"].split()
    right_tokens = row["right"].split()

    result = {
        "shared_candidate": "",
        "separate_candidate": "",
        "shared_exact_anywhere": False,
        "separate_exact_anywhere": False,
        "shared_same_address": False,
        "separate_same_address": False
    }

    if (
        len(left_tokens) < 2
        or len(right_tokens) != 2
    ):
        return pd.Series(result)

    shared_surname = left_tokens[0]

    right_1 = right_tokens[0]
    right_2 = right_tokens[1]

    # Hypothesis A:
    # right side is FIRST MIDDLE and inherits left surname.
    shared_candidate = (
        f"{shared_surname}, "
        f"{right_1} {right_2}"
    )

    # Hypothesis B:
    # right side is LAST FIRST.
    separate_candidate = (
        f"{right_1}, "
        f"{right_2}"
    )

    result[
        "shared_candidate"
    ] = shared_candidate

    result[
        "separate_candidate"
    ] = separate_candidate

    shared_parsed = (
        resolver._parse_person_name(
            shared_candidate
        )
    )

    separate_parsed = (
        resolver._parse_person_name(
            separate_candidate
        )
    )

    address = row.get(
        "associated_address",
        ""
    )

    if shared_parsed is not None:

        shared_matches = person_evidence[
            (person_evidence["last_name"]
             == shared_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == shared_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == shared_parsed["middle_tokens"])
        ]

        result[
            "shared_exact_anywhere"
        ] = len(shared_matches) > 0

        result[
            "shared_same_address"
        ] = (
            (
                shared_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    if separate_parsed is not None:

        separate_matches = person_evidence[
            (person_evidence["last_name"]
             == separate_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == separate_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == separate_parsed["middle_tokens"])
        ]

        result[
            "separate_exact_anywhere"
        ] = len(separate_matches) > 0

        result[
            "separate_same_address"
        ] = (
            (
                separate_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    return pd.Series(result)

In [ ]:
right_two_full_with_address = (
    right_two_full_tokens
    .merge(
        remaining_joint[
            [
                "entity_name",
                "associated_address"
            ]
        ].drop_duplicates(),
        on="entity_name",
        how="left"
    )
)

right_two_full_with_address

In [ ]:
context_evidence = pd.concat(
    [
        right_two_full_with_address
        .reset_index(drop=True),

        right_two_full_with_address
        .reset_index(drop=True)
        .apply(
            evaluate_two_full_token_joint,
            axis=1
        )
    ],
    axis=1
)

context_evidence

In [ ]:
context_evidence_by_name = (
    context_evidence
    .groupby(
        "entity_name",
        as_index=False
    )
    .agg(
        shared_candidate=(
            "shared_candidate",
            "first"
        ),
        separate_candidate=(
            "separate_candidate",
            "first"
        ),

        shared_exact_anywhere=(
            "shared_exact_anywhere",
            "any"
        ),
        separate_exact_anywhere=(
            "separate_exact_anywhere",
            "any"
        ),

        shared_same_address=(
            "shared_same_address",
            "any"
        ),
        separate_same_address=(
            "separate_same_address",
            "any"
        ),

        address_count=(
            "associated_address",
            "nunique"
        )
    )
)

In [ ]:
print(
    "Unique Joint names in contextual analysis:",
    len(context_evidence_by_name)
)

In [ ]:
pd.crosstab(
    context_evidence_by_name[
        "shared_same_address"
    ],
    context_evidence_by_name[
        "separate_same_address"
    ],
    margins=True
)

In [ ]:
same_address_evidence = (
    context_evidence_by_name[
        context_evidence_by_name[
            [
                "shared_same_address",
                "separate_same_address"
            ]
        ].any(axis=1)
    ]
    [
        [
            "entity_name",
            "shared_candidate",
            "separate_candidate",
            "shared_same_address",
            "separate_same_address",
            "shared_exact_anywhere",
            "separate_exact_anywhere",
            "address_count"
        ]
    ]
    .sort_values(
        [
            "shared_same_address",
            "separate_same_address",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
)

same_address_evidence

In [ ]:
def classify_contextual_evidence(row):

    shared_same = row[
        "shared_same_address"
    ]

    separate_same = row[
        "separate_same_address"
    ]

    shared_any = row[
        "shared_exact_anywhere"
    ]

    separate_any = row[
        "separate_exact_anywhere"
    ]


    # ------------------------------------------------------
    # Strongest evidence: same address
    # ------------------------------------------------------

    if shared_same and not separate_same:
        return "Shared surname - same address"

    if separate_same and not shared_same:
        return "Separate surnames - same address"

    if shared_same and separate_same:
        return "Conflict - both same address"


    # ------------------------------------------------------
    # Secondary evidence: exact candidate elsewhere
    # ------------------------------------------------------

    if shared_any and not separate_any:
        return "Shared surname - elsewhere"

    if separate_any and not shared_any:
        return "Separate surnames - elsewhere"

    if shared_any and separate_any:
        return "Conflict - both elsewhere"


    # ------------------------------------------------------
    # No contextual evidence
    # ------------------------------------------------------

    return "No contextual evidence"

In [ ]:
context_evidence_by_name[
    "context_decision"
] = (
    context_evidence_by_name
    .apply(
        classify_contextual_evidence,
        axis=1
    )
)

In [ ]:
context_evidence_by_name[
    "context_decision"
].value_counts()

## Addressing cases of same name similar addresses

In [ ]:
STREET_SUFFIXES = {
    "ST",
    "AVE",
    "RD",
    "DR",
    "BLVD",
    "LN",
    "CT",
    "CIR",
    "PL",
    "PKWY",
    "TER",
    "HWY",
    "WAY",
    "TRL"
}

UNIT_MARKERS = {
    "APT",
    "STE",
    "UNIT",
    "BLDG",
    "FL",
    "RM"
}


def parse_address_suffix_for_audit(address):
    """
    Break a cleaned address into components needed to detect
    cases where a street suffix is present in one representation
    and absent in another.

    This is for analysis only and does not modify the address.
    """

    empty_result = pd.Series({
        "street_base": None,
        "street_suffix": None,
        "unit": None,
        "locality": None
    })

    if pd.isna(address):
        return empty_result

    address = str(address).strip()

    if not address:
        return empty_result

    parts = [
        part.strip()
        for part in address.split(",")
    ]

    # Require at least:
    #
    # street, locality...
    if len(parts) < 2:
        return empty_result

    street_line = parts[0]

    # Don't try to interpret PO Boxes as street addresses.
    if street_line.startswith("PO BOX"):
        return empty_result

    locality = ", ".join(parts[1:])

    tokens = street_line.split()

    if len(tokens) < 2:
        return empty_result


    # ----------------------------------------------------------
    # Separate unit information
    # ----------------------------------------------------------

    unit_index = None

    for i, token in enumerate(tokens):

        if (
            token in UNIT_MARKERS
            or token.startswith("#")
        ):
            unit_index = i
            break

    if unit_index is not None:

        street_tokens = tokens[:unit_index]
        unit_tokens = tokens[unit_index:]

    else:

        street_tokens = tokens
        unit_tokens = []


    if len(street_tokens) < 2:
        return empty_result


    # Require something resembling a street number.
    if not any(
        character.isdigit()
        for character in street_tokens[0]
    ):
        return empty_result


    # ----------------------------------------------------------
    # Identify street suffix
    # ----------------------------------------------------------

    if street_tokens[-1] in STREET_SUFFIXES:

        street_suffix = street_tokens[-1]

        street_base_tokens = (
            street_tokens[:-1]
        )

    else:

        street_suffix = ""

        street_base_tokens = street_tokens


    if not street_base_tokens:
        return empty_result


    return pd.Series({
        "street_base":
            " ".join(street_base_tokens),

        "street_suffix":
            street_suffix,

        "unit":
            " ".join(unit_tokens),

        "locality":
            locality
    })

In [ ]:
address_suffix_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
)

parsed_addresses = (
    address_suffix_audit[
        "canonical_address"
    ]
    .apply(
        parse_address_suffix_for_audit
    )
)

address_suffix_audit = pd.concat(
    [
        address_suffix_audit.reset_index(
            drop=True
        ),

        parsed_addresses.reset_index(
            drop=True
        )
    ],
    axis=1
)

In [ ]:
address_suffix_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "street_base",
    "unit",
    "locality"
]

for _, group in (
    address_suffix_audit
    .dropna(
        subset=[
            "street_base",
            "street_suffix",
            "locality"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    suffixes = set(
        group["street_suffix"]
    )

    nonblank_suffixes = {
        suffix
        for suffix in suffixes
        if suffix
    }

    # Require:
    #
    #   one suffix-free version
    #   +
    #   exactly one known suffix
    #
    # Example:
    #
    #   214 MARSHALL
    #   214 MARSHALL ST
    #
    if (
        "" not in suffixes
        or len(nonblank_suffixes) != 1
    ):
        continue

    address_suffix_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "street_base":
            group["street_base"].iloc[0],

        "locality":
            group["locality"].iloc[0],

        "unit":
            group["unit"].iloc[0],

        "known_suffix":
            next(iter(nonblank_suffixes)),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "addresses":
            tuple(
                group["canonical_address"]
            ),

        "entity_count":
            len(group)
    })


address_suffix_candidates = pd.DataFrame(
    address_suffix_candidates
)

In [ ]:
print(
    "Candidate missing-street-suffix groups:",
    len(address_suffix_candidates)
)

address_suffix_candidates

In [ ]:
entities["occurrence_count"].sum()

In [ ]:
entities[
    entities["canonical_name"]
    .str.contains(
        "WIDGER JAMES LEROY",
        regex=False,
        na=False
    )
]

In [ ]:
entities.head()
entities.info()

In [ ]:
print(
    "Unique source occurrences:",
    entity_occurrences["source_occurrence_id"].nunique()
)

print(
    "Entity occurrences after Joint expansion:",
    len(entity_occurrences)
)

print(
    "Unique occurrence IDs:",
    entity_occurrences["occurrence_id"].nunique()
)

print(
    "Sum of final entity occurrence counts:",
    entities["occurrence_count"].sum()
)

## Suffix merge change analysis

In [ ]:
entities[
    entities["canonical_name"]
    .str.contains(
        "DERRING",
        regex=False,
        na=False
    )
][
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
]

In [ ]:
entities[
    entities["canonical_name"]
    .str.contains(
        "LINDSEY, WILLIAM B",
        regex=False,
        na=False
    )
][
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
]

In [ ]:
entities["occurrence_count"].sum()
# 12058

## Address zip code auditing

In [ ]:
import re


def parse_zip_variant_for_audit(address):
    """
    Extract the address body and ZIP components for auditing
    5-digit ZIP versus ZIP+4 variants.

    Does not modify the original address.
    """

    if pd.isna(address):
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    address = str(address).strip()

    if not address:
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    match = re.search(
        r"^(.*\D)(\d{5})(?:-(\d{4}))?$",
        address
    )

    if match is None:
        return pd.Series({
            "address_without_zip": None,
            "zip5": None,
            "zip4": None
        })

    return pd.Series({
        "address_without_zip":
            match.group(1).strip(),

        "zip5":
            match.group(2),

        "zip4":
            match.group(3) or ""
    })

In [ ]:
zip_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
    .reset_index(drop=True)
)

zip_parts = (
    zip_audit["canonical_address"]
    .apply(parse_zip_variant_for_audit)
)

zip_audit = pd.concat(
    [
        zip_audit,
        zip_parts
    ],
    axis=1
)

In [ ]:
zip_variant_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "address_without_zip",
    "zip5"
]

for _, group in (
    zip_audit
    .dropna(
        subset=[
            "address_without_zip",
            "zip5"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    zip4_values = set(
        group["zip4"]
    )

    nonblank_zip4 = {
        value
        for value in zip4_values
        if value
    }

    # Require:
    #
    #     49048
    #     49048-4827
    #
    # but reject a group containing conflicting ZIP+4 values.
    if (
        "" not in zip4_values
        or len(nonblank_zip4) != 1
    ):
        continue

    zip_variant_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "address_without_zip":
            group["address_without_zip"].iloc[0],

        "zip5":
            group["zip5"].iloc[0],

        "known_zip4":
            next(iter(nonblank_zip4)),

        "addresses":
            tuple(
                group["canonical_address"]
                .drop_duplicates()
            ),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "entity_count":
            len(group)
    })


zip_variant_candidates = pd.DataFrame(
    zip_variant_candidates
)

print(
    "Candidate ZIP+4 variant groups:",
    len(zip_variant_candidates)
)

zip_variant_candidates

## Testing collapse_index_groups change

In [ ]:
test_entities = pd.DataFrame({
    "canonical_name": [
        "SMITH, JOHN",
        "SMITH, JOHN A",
        "SMITH, JOHN ALAN"
    ],

    "entity_type": [
        "Person",
        "Person",
        "Person"
    ],

    "canonical_address": [
        "123 MAIN ST, KALAMAZOO, MI 49001",
        "123 MAIN ST, KALAMAZOO, MI 49001",
        "123 MAIN ST, KALAMAZOO, MI 49001"
    ],

    "organization_type": [
        None,
        None,
        None
    ],

    "organization_classification_confidence": [
        None,
        None,
        None
    ],

    "occurrence_count": [
        1,
        1,
        1
    ],

    "_occurrence_ids": [
        (1,),
        (2,),
        (3,)
    ]
})